In [1]:
# ============================================================
# FREIGHT-MNet: Download BTS FMI County-to-County Truck Travel Times
# Years: 2018–2024
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\02_fmi
#
# What this downloads:
#   Public BTS FMI annual county-pair truck travel-time CSV files.
#
# Output:
#   - raw annual CSV
#   - annual Parquet file for faster future processing
#   - dataset metadata JSON
#   - columns metadata JSON
#   - preview CSV
#   - manifest CSV
#   - download log JSONL
# ============================================================

from pathlib import Path
import os
import sys
import json
import time
import shutil
import hashlib
import requests
import pandas as pd
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Optional but recommended: Polars is faster for large CSV -> Parquet conversion.
try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
FMI_ROOT = DATA_ROOT / "02_fmi"
METADATA_ROOT = FMI_ROOT / "_metadata"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

MANIFEST_PATH = MANIFEST_ROOT / "bts_fmi_download_manifest.csv"
LOG_PATH = MANIFEST_ROOT / "bts_fmi_download_log.jsonl"

# Keep raw CSV. Convert each year to Parquet too, which is much faster for later processing.
CONVERT_TO_PARQUET = True

# If True, existing CSV files will be checked and skipped when they look valid.
# If False, it will redownload everything.
SKIP_EXISTING_VALID_FILES = True

# If True, code will compute sha256 hash for each CSV.
# This is useful but slower for large files.
COMPUTE_SHA256 = False

# BTS / Socrata dataset IDs.
# These are the annual public FMI county-to-county truck travel-time files.
FMI_DATASETS = {
    2018: {
        "dataset_id": "xx4g-5dg2",
        "expected_unique_pairs_million": 2.08,
    },
    2019: {
        "dataset_id": "sn4k-eiea",
        "expected_unique_pairs_million": 4.13,
    },
    2020: {
        "dataset_id": "dggd-bg3y",
        "expected_unique_pairs_million": 4.43,
    },
    2021: {
        "dataset_id": "mayv-2qfz",
        "expected_unique_pairs_million": 4.25,
    },
    2022: {
        "dataset_id": "d7b8-pmxm",
        "expected_unique_pairs_million": 3.81,
    },
    2023: {
        "dataset_id": "ez58-m3b4",
        "expected_unique_pairs_million": 3.64,
    },
    2024: {
        "dataset_id": "uta5-4eu5",
        "expected_unique_pairs_million": 3.58,
    },
}

# data.bts.gov is the user-facing BTS data domain.
# data.transportation.gov is used as fallback because Socrata sometimes exposes DOT data there.
SOCRATA_DOMAINS = [
    "https://data.bts.gov",
    "https://data.transportation.gov",
]

# Create directories.
FMI_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

for year in FMI_DATASETS:
    (FMI_ROOT / str(year)).mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("BTS FMI downloader")
print("Python executable:", sys.executable)
print("DATA_ROOT:", DATA_ROOT)
print("FMI_ROOT:", FMI_ROOT)
print("HAS_POLARS:", HAS_POLARS)
print("=" * 100)

# ------------------------------------------------------------
# 1. HTTP session with retries
# ------------------------------------------------------------

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "FREIGHT-MNet academic research downloader "
            "(public BTS FMI county-to-county truck travel-time data)"
        )
    }
)

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=2.0,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)


# ------------------------------------------------------------
# 2. Utility functions
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 3)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def is_probably_valid_csv(path: Path, min_size_mb: float = 1.0) -> bool:
    """
    Quick validation:
    - file exists
    - file size > min_size_mb
    - pandas can read first few rows
    """
    if not path.exists():
        return False
    if file_size_mb(path) < min_size_mb:
        return False
    try:
        preview = pd.read_csv(path, nrows=3)
        if preview.shape[1] == 0:
            return False
        return True
    except Exception:
        return False


def build_urls(domain: str, dataset_id: str) -> dict:
    return {
        "csv": f"{domain}/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD",
        "metadata": f"{domain}/api/views/{dataset_id}.json",
        "columns": f"{domain}/api/views/{dataset_id}/columns.json",
        "count": f"{domain}/resource/{dataset_id}.json?$select=count(*)",
        "about": f"{domain}/d/{dataset_id}",
    }


def get_first_working_url(dataset_id: str, kind: str) -> tuple[str, str]:
    """
    Try data.bts.gov first, then data.transportation.gov.
    Returns: (domain, url)
    """
    last_error = None

    for domain in SOCRATA_DOMAINS:
        url = build_urls(domain, dataset_id)[kind]
        try:
            if kind == "csv":
                # HEAD sometimes fails on Socrata, so use a streamed GET and read no content.
                r = session.get(url, stream=True, timeout=60)
            else:
                r = session.get(url, timeout=60)

            if r.status_code < 400:
                r.close()
                return domain, url

            last_error = f"{url} -> HTTP {r.status_code}"
            r.close()

        except Exception as e:
            last_error = f"{url} -> {repr(e)}"

    raise RuntimeError(f"No working {kind} URL found for dataset {dataset_id}. Last error: {last_error}")


def download_streaming(url: str, out_path: Path, desc: str, overwrite: bool = False) -> Path:
    """
    Streaming download with .part temporary file.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not overwrite and is_probably_valid_csv(out_path):
        print(f"[skip existing valid file] {out_path} ({file_size_mb(out_path)} MB)")
        log_event(
            {
                "status": "exists_valid",
                "url": url,
                "path": str(out_path),
                "size_mb": file_size_mb(out_path),
            }
        )
        return out_path

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    if part_path.exists():
        part_path.unlink()

    print("\n" + "-" * 100)
    print("[download]", desc)
    print("URL:", url)
    print("OUT:", out_path)
    print("-" * 100)

    with session.get(url, stream=True, timeout=300) as r:
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code} while downloading {url}")

        total = int(r.headers.get("content-length", 0))
        with part_path.open("wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=desc,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    part_path.replace(out_path)

    if not is_probably_valid_csv(out_path):
        raise RuntimeError(f"Downloaded file does not look like a valid CSV: {out_path}")

    log_event(
        {
            "status": "downloaded",
            "url": url,
            "path": str(out_path),
            "size_mb": file_size_mb(out_path),
        }
    )

    print(f"[saved] {out_path} ({file_size_mb(out_path)} MB)")
    return out_path


def save_json_from_url(url: str, out_path: Path, desc: str) -> bool:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        r = session.get(url, timeout=120)
        if r.status_code >= 400:
            print(f"[metadata warn] {desc}: HTTP {r.status_code}")
            log_event(
                {
                    "status": "metadata_failed",
                    "url": url,
                    "path": str(out_path),
                    "reason": f"HTTP {r.status_code}",
                }
            )
            return False

        data = r.json()
        out_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"[metadata saved] {out_path}")
        log_event(
            {
                "status": "metadata_saved",
                "url": url,
                "path": str(out_path),
            }
        )
        return True

    except Exception as e:
        print(f"[metadata warn] {desc}: {e}")
        log_event(
            {
                "status": "metadata_failed",
                "url": url,
                "path": str(out_path),
                "reason": repr(e),
            }
        )
        return False


def get_socrata_row_count(url: str) -> int | None:
    try:
        r = session.get(url, timeout=120)
        if r.status_code >= 400:
            return None
        data = r.json()
        if isinstance(data, list) and len(data) > 0:
            # Socrata returns [{"count": "123"}]
            val = list(data[0].values())[0]
            return int(val)
        return None
    except Exception:
        return None


def write_preview_and_schema(csv_path: Path, out_dir: Path, year: int) -> dict:
    """
    Save:
    - preview first 20 rows
    - schema inferred by pandas from first rows
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    preview_path = out_dir / f"bts_fmi_{year}_preview_first20.csv"
    schema_path = out_dir / f"bts_fmi_{year}_schema_preview.csv"

    preview = pd.read_csv(csv_path, nrows=20)
    preview.to_csv(preview_path, index=False, encoding="utf-8-sig")

    schema_df = pd.DataFrame(
        {
            "column": preview.columns,
            "pandas_preview_dtype": [str(t) for t in preview.dtypes],
        }
    )
    schema_df.to_csv(schema_path, index=False, encoding="utf-8-sig")

    return {
        "preview_path": str(preview_path),
        "schema_preview_path": str(schema_path),
        "preview_columns": list(preview.columns),
        "preview_n_columns": len(preview.columns),
    }


def convert_csv_to_parquet(csv_path: Path, parquet_path: Path, year: int) -> dict:
    """
    Convert annual CSV to Parquet.
    Uses Polars if available. Falls back to pandas if Polars is unavailable.
    """
    parquet_path = Path(parquet_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    if parquet_path.exists() and parquet_path.stat().st_size > 0:
        print(f"[skip existing parquet] {parquet_path} ({file_size_mb(parquet_path)} MB)")

        if HAS_POLARS:
            try:
                n_rows = pl.scan_parquet(str(parquet_path)).select(pl.len()).collect().item()
            except Exception:
                n_rows = None
        else:
            n_rows = None

        return {
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "row_count_from_conversion": n_rows,
            "conversion_status": "exists",
        }

    print("\n" + "-" * 100)
    print(f"[convert] CSV -> Parquet for {year}")
    print("CSV:", csv_path)
    print("PARQUET:", parquet_path)
    print("-" * 100)

    if HAS_POLARS:
        # For these annual FMI files, reading one year at a time should be manageable.
        df = pl.read_csv(
            str(csv_path),
            infer_schema_length=10000,
            ignore_errors=False,
            try_parse_dates=False,
        )
        n_rows = df.height
        n_cols = len(df.columns)
        df.write_parquet(str(parquet_path), compression="zstd")
        del df

    else:
        # Slower fallback.
        df = pd.read_csv(csv_path)
        n_rows = len(df)
        n_cols = len(df.columns)
        df.to_parquet(parquet_path, index=False)
        del df

    print(f"[parquet saved] {parquet_path} ({file_size_mb(parquet_path)} MB)")

    return {
        "parquet_path": str(parquet_path),
        "parquet_size_mb": file_size_mb(parquet_path),
        "row_count_from_conversion": n_rows,
        "n_columns_from_conversion": n_cols,
        "conversion_status": "converted",
    }


# ------------------------------------------------------------
# 3. Download all FMI annual files
# ------------------------------------------------------------

manifest_rows = []

for year, info in FMI_DATASETS.items():
    dataset_id = info["dataset_id"]
    year_dir = FMI_ROOT / str(year)
    meta_year_dir = METADATA_ROOT / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)
    meta_year_dir.mkdir(parents=True, exist_ok=True)

    csv_path = year_dir / f"bts_fmi_county_travel_times_{year}_{dataset_id}.csv"
    parquet_path = year_dir / f"bts_fmi_county_travel_times_{year}_{dataset_id}.parquet"

    print("\n" + "=" * 100)
    print(f"YEAR {year} | DATASET ID {dataset_id}")
    print("=" * 100)

    # Find working domain and URL.
    domain, csv_url = get_first_working_url(dataset_id, "csv")
    urls = build_urls(domain, dataset_id)

    print("Working Socrata domain:", domain)

    # Save metadata and columns metadata.
    save_json_from_url(
        urls["metadata"],
        meta_year_dir / f"bts_fmi_{year}_{dataset_id}_metadata.json",
        desc=f"{year} metadata",
    )
    save_json_from_url(
        urls["columns"],
        meta_year_dir / f"bts_fmi_{year}_{dataset_id}_columns.json",
        desc=f"{year} columns metadata",
    )

    # Get Socrata row count if API supports it.
    socrata_row_count = get_socrata_row_count(urls["count"])
    print("Socrata API row count:", socrata_row_count)

    # Download CSV.
    download_start = time.time()

    csv_status = "unknown"
    try:
        csv_path = download_streaming(
            csv_url,
            csv_path,
            desc=f"BTS FMI {year}",
            overwrite=not SKIP_EXISTING_VALID_FILES,
        )
        csv_status = "ok"
    except Exception as e:
        # Try fallback domain if first domain failed.
        print(f"[warn] primary download failed for {year}: {e}")
        log_event(
            {
                "status": "download_failed_primary",
                "year": year,
                "dataset_id": dataset_id,
                "url": csv_url,
                "reason": repr(e),
            }
        )

        fallback_domains = [d for d in SOCRATA_DOMAINS if d != domain]
        fallback_ok = False

        for fallback_domain in fallback_domains:
            fallback_url = build_urls(fallback_domain, dataset_id)["csv"]
            try:
                csv_path = download_streaming(
                    fallback_url,
                    csv_path,
                    desc=f"BTS FMI {year} fallback",
                    overwrite=True,
                )
                domain = fallback_domain
                urls = build_urls(domain, dataset_id)
                csv_status = "ok_fallback"
                fallback_ok = True
                break
            except Exception as e2:
                print(f"[warn] fallback failed for {year} on {fallback_domain}: {e2}")

        if not fallback_ok:
            raise RuntimeError(f"All download attempts failed for year {year}, dataset {dataset_id}")

    download_seconds = round(time.time() - download_start, 2)

    # Preview and schema.
    preview_info = write_preview_and_schema(csv_path, meta_year_dir, year)
    print("Preview columns:")
    for c in preview_info["preview_columns"]:
        print("  -", c)

    # Convert to Parquet.
    parquet_info = {}
    if CONVERT_TO_PARQUET:
        try:
            parquet_info = convert_csv_to_parquet(csv_path, parquet_path, year)
        except Exception as e:
            print(f"[warn] Parquet conversion failed for {year}: {e}")
            parquet_info = {
                "parquet_path": str(parquet_path),
                "parquet_size_mb": None,
                "row_count_from_conversion": None,
                "conversion_status": f"failed: {repr(e)}",
            }

    # Optional SHA256.
    if COMPUTE_SHA256:
        csv_sha256 = sha256_file(csv_path)
    else:
        csv_sha256 = None

    row = {
        "year": year,
        "dataset_id": dataset_id,
        "expected_unique_pairs_million": info["expected_unique_pairs_million"],
        "socrata_domain_used": domain,
        "csv_url": csv_url,
        "about_url": urls["about"],
        "csv_path": str(csv_path),
        "csv_size_mb": file_size_mb(csv_path),
        "csv_status": csv_status,
        "csv_sha256": csv_sha256,
        "download_seconds": download_seconds,
        "socrata_api_row_count": socrata_row_count,
        **preview_info,
        **parquet_info,
    }

    manifest_rows.append(row)

    # Save manifest after each year so progress is not lost.
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
    print(f"[manifest updated] {MANIFEST_PATH}")

# ------------------------------------------------------------
# 4. Final manifest and summary
# ------------------------------------------------------------

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\n" + "=" * 100)
print("BTS FMI DOWNLOAD COMPLETE")
print("=" * 100)
print("FMI root:", FMI_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Log:", LOG_PATH)
print("=" * 100)

display_cols = [
    "year",
    "dataset_id",
    "csv_status",
    "csv_size_mb",
    "parquet_size_mb",
    "socrata_api_row_count",
    "row_count_from_conversion",
    "download_seconds",
    "csv_path",
]
existing_cols = [c for c in display_cols if c in manifest.columns]
display(manifest[existing_cols])

print("\nNext files to check:")
for year, info in FMI_DATASETS.items():
    dataset_id = info["dataset_id"]
    print(FMI_ROOT / str(year) / f"bts_fmi_county_travel_times_{year}_{dataset_id}.csv")
    if CONVERT_TO_PARQUET:
        print(FMI_ROOT / str(year) / f"bts_fmi_county_travel_times_{year}_{dataset_id}.parquet")

BTS FMI downloader
Python executable: D:\apphome\anaconda3\python.exe
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
FMI_ROOT: E:\NetworkOptimization\pythonProject1\Data\02_fmi
HAS_POLARS: False

YEAR 2018 | DATASET ID xx4g-5dg2
Working Socrata domain: https://data.bts.gov
[metadata saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\_metadata\2018\bts_fmi_2018_xx4g-5dg2_metadata.json
[metadata saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\_metadata\2018\bts_fmi_2018_xx4g-5dg2_columns.json
Socrata API row count: 2080775

----------------------------------------------------------------------------------------------------
[download] BTS FMI 2018
URL: https://data.bts.gov/api/views/xx4g-5dg2/rows.csv?accessType=DOWNLOAD
OUT: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.csv
----------------------------------------------------------------------------------------------------


BTS FMI 2018: 0.00B [00:00, ?B/s]

[saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.csv (131.977 MB)
Preview columns:
  - Origin CTFIPS
  - Origin State
  - Origin County
  - Destination CTFIPS
  - Destination State
  - Destination County
  - Year
  - Crossings
  - 25th Percentile (minutes)
  - 50th Percentile (minutes)
  - 75th Percentile (minutes)

----------------------------------------------------------------------------------------------------
[convert] CSV -> Parquet for 2018
CSV: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.csv
PARQUET: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.parquet
----------------------------------------------------------------------------------------------------
[parquet saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.parquet (46.312 MB)
[manifest updated] E:\NetworkOpti

BTS FMI 2019: 0.00B [00:00, ?B/s]

[saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.csv (261.0 MB)
Preview columns:
  - Origin CTFIPS
  - Origin State
  - Origin County
  - Destination CTFIPS
  - Destination State
  - Destination County
  - Year
  - Movements
  - 25th Percentile (minutes)
  - 50th Percentile (minutes)
  - 75th Percentile (minutes)

----------------------------------------------------------------------------------------------------
[convert] CSV -> Parquet for 2019
CSV: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.csv
PARQUET: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.parquet
----------------------------------------------------------------------------------------------------
[parquet saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.parquet (94.943 MB)
[manifest updated] E:\NetworkOptimi

...

[saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2023\bts_fmi_county_travel_times_2023_ez58-m3b4.csv (231.176 MB)
Preview columns:
  - Origin CTFIPS
  - Origin State
  - Origin County
  - Destination CTFIPS
  - Destination State
  - Destination County
  - Year
  - Movements
  - 25th Percentile (minutes)
  - 50th Percentile (minutes)
  - 75th Percentile (minutes)

----------------------------------------------------------------------------------------------------
[convert] CSV -> Parquet for 2023
CSV: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2023\bts_fmi_county_travel_times_2023_ez58-m3b4.csv
PARQUET: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2023\bts_fmi_county_travel_times_2023_ez58-m3b4.parquet
----------------------------------------------------------------------------------------------------
[parquet saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2023\bts_fmi_county_travel_times_2023_ez58-m3b4.parquet (83.284 MB)
[manifest updated] E:\NetworkOpti

BTS FMI 2024: 0.00B [00:00, ?B/s]

[saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2024\bts_fmi_county_travel_times_2024_uta5-4eu5.csv (227.27 MB)
Preview columns:
  - Origin CTFIPS
  - Origin State
  - Origin County
  - Destination CTFIPS
  - Destination State
  - Destination County
  - Year
  - Movements
  - 25th Percentile (minutes)
  - 50th Percentile (minutes)
  - 75th Percentile (minutes)

----------------------------------------------------------------------------------------------------
[convert] CSV -> Parquet for 2024
CSV: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2024\bts_fmi_county_travel_times_2024_uta5-4eu5.csv
PARQUET: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2024\bts_fmi_county_travel_times_2024_uta5-4eu5.parquet
----------------------------------------------------------------------------------------------------
[parquet saved] E:\NetworkOptimization\pythonProject1\Data\02_fmi\2024\bts_fmi_county_travel_times_2024_uta5-4eu5.parquet (82.404 MB)
[manifest updated] E:\NetworkOptim

,year,dataset_id,csv_status,csv_size_mb,parquet_size_mb,socrata_api_row_count,row_count_from_conversion,download_seconds,csv_path
0,2018,xx4g-5dg2,ok,131.977,46.312,2080775,2080775,46.10,E:\NetworkOptimization\pythonProject1\Data\02_...
1,2019,sn4k-eiea,ok,261.000,94.943,4130615,4130615,90.53,E:\NetworkOptimization\pythonProject1\Data\02_...
2,2020,dggd-bg3y,ok,281.613,102.526,4429137,4429137,424.03,E:\NetworkOptimization\pythonProject1\Data\02_...
3,2021,mayv-2qfz,ok,270.129,98.444,4248604,4248604,663.67,E:\NetworkOptimization\pythonProject1\Data\02_...
4,2022,d7b8-pmxm,ok,241.886,87.690,3807134,3807134,77.15,E:\NetworkOptimization\pythonProject1\Data\02_...
5,2023,ez58-m3b4,ok,231.176,83.284,3642150,3642150,482.48,E:\NetworkOptimization\pythonProject1\Data\02_...
6,2024,uta5-4eu5,ok,227.270,82.404,3585730,3585730,176.66,E:\NetworkOptimization\pythonProject1\Data\02_...



Next files to check:
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.csv
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.parquet
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.csv
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.parquet
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2020\bts_fmi_county_travel_times_2020_dggd-bg3y.csv
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2020\bts_fmi_county_travel_times_2020_dggd-bg3y.parquet
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2021\bts_fmi_county_travel_times_2021_mayv-2qfz.csv
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2021\bts_fmi_county_travel_times_2021_mayv-2qfz.parquet
E:\NetworkOptimization\pythonProject1\Data\02_fmi\2022\bts_fmi_county_travel_times_2022_d7b8-pmxm.csv
E:\NetworkOptimization\pythonProject1\Data\0

In [3]:
# ============================================================
# FREIGHT-MNet: NTAD / ArcGIS Geospatial Downloader v2
#
# FIXED VERSION:
#   - Uses ArcGIS f=json instead of f=geojson
#   - Converts Esri geometry to Shapely manually
#   - No infinite split retry
#   - Fully resumable: rerun the cell and completed chunks are skipped
#
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial
#
# Downloads:
#   1. FAF5 Regions
#   2. FAF5 Network Links
#   3. FAF5 Network Nodes
#   4. NARN Lines
#   5. NARN Nodes
#   6. Intermodal Freight Facilities Rail TOFC/COFC
# ============================================================

from pathlib import Path
import json
import time
import traceback
import math
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString, Polygon, MultiPolygon
from shapely.validation import make_valid
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ------------------------------------------------------------
# 0. Paths and settings
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
NTAD_ROOT = DATA_ROOT / "03_ntad_geospatial"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

NTAD_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

GLOBAL_MANIFEST_PATH = MANIFEST_ROOT / "ntad_arcgis_download_manifest_v2.csv"
GLOBAL_LOG_PATH = MANIFEST_ROOT / "ntad_arcgis_download_log_v2.jsonl"

# Safe batch size. 1000 is more stable than 2000 on some ArcGIS hosted layers.
DEFAULT_BATCH_SIZE = 1000

# Rerun-safe.
RESUME = True

# Keep raw Esri JSON chunks for reproducibility.
SAVE_RAW_JSON = True

# Build combined file for small layers only.
CREATE_SINGLE_FILE_FOR_SMALL_LAYERS = True
SINGLE_FILE_MAX_FEATURES = 10000

# Debug only. Set to None for real full download.
MAX_OBJECTS_PER_LAYER = None

# If True, failed chunks are retried once with smaller sub-batches.
RETRY_FAILED_CHUNKS_WITH_SMALLER_BATCH = True
SMALL_RETRY_BATCH_SIZE = 100

# ------------------------------------------------------------
# 1. Layers
# ------------------------------------------------------------

LAYERS = {
    "faf_regions": {
        "description": "FAF5 Regions / FAF zone polygons",
        "out_subdir": "faf_regions",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_Freight_Analysis_Framework_Regions/FeatureServer/0",
    },
    "faf_network_links": {
        "description": "FAF5 Network Links / highway freight network polyline topology",
        "out_subdir": "faf_network_links",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_Freight_Analysis_Framework_Network_Links/FeatureServer/0",
    },
    "faf_network_nodes": {
        "description": "FAF5 Network Nodes / highway freight network node topology",
        "out_subdir": "faf_network_nodes",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_Freight_Analysis_Framework_Network_Nodes/FeatureServer/0",
    },
    "narn_lines": {
        "description": "North American Rail Network Lines / rail line polyline topology",
        "out_subdir": "narn_lines",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_North_American_Rail_Network_Lines/FeatureServer/0",
    },
    "narn_nodes": {
        "description": "North American Rail Network Nodes / rail node topology",
        "out_subdir": "narn_nodes",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_North_American_Rail_Network_Nodes/FeatureServer/0",
    },
    "intermodal_rail_tofc_cofc": {
        "description": "Intermodal Freight Facilities Rail TOFC/COFC / truck-rail transfer terminals",
        "out_subdir": "intermodal_rail_tofc_cofc",
        "url": "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_Intermodal_Freight_Facilities_Rail_TOFC_COFC/FeatureServer/0",
    },
}

# ------------------------------------------------------------
# 2. HTTP session
# ------------------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "FREIGHT-MNet academic downloader v2; public USDOT/NTAD ArcGIS REST data"
})

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=1.5,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------

def log_event(record: Dict[str, Any]) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with GLOBAL_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def request_json(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    method: str = "GET",
    timeout: int = 300,
) -> Dict[str, Any]:
    if method.upper() == "POST":
        r = session.post(url, data=params, timeout=timeout)
    else:
        r = session.get(url, params=params, timeout=timeout)

    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    try:
        data = r.json()
    except Exception as e:
        raise RuntimeError(f"Could not parse JSON. First 500 chars: {r.text[:500]}") from e

    if isinstance(data, dict) and "error" in data:
        raise RuntimeError(f"ArcGIS error: {data['error']}")

    return data


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 3)


def get_metadata(layer_url: str) -> Dict[str, Any]:
    return request_json(layer_url, params={"f": "json"}, method="GET", timeout=120)


def get_oid_field(metadata: Dict[str, Any]) -> str:
    if metadata.get("objectIdField"):
        return metadata["objectIdField"]
    if metadata.get("objectIdFieldName"):
        return metadata["objectIdFieldName"]
    for f in metadata.get("fields", []):
        if f.get("type") == "esriFieldTypeOID":
            return f["name"]
    return "OBJECTID"


def get_count(layer_url: str) -> int:
    data = request_json(
        layer_url + "/query",
        params={
            "where": "1=1",
            "returnCountOnly": "true",
            "f": "json",
        },
        method="GET",
        timeout=120,
    )
    return int(data["count"])


def get_object_ids(layer_url: str) -> List[int]:
    data = request_json(
        layer_url + "/query",
        params={
            "where": "1=1",
            "returnIdsOnly": "true",
            "f": "json",
        },
        method="GET",
        timeout=300,
    )
    object_ids = sorted(int(x) for x in data["objectIds"])
    if MAX_OBJECTS_PER_LAYER is not None:
        object_ids = object_ids[:MAX_OBJECTS_PER_LAYER]
    return object_ids


def chunk_list(xs: List[int], size: int) -> List[List[int]]:
    return [xs[i:i + size] for i in range(0, len(xs), size)]


def sanitize_attributes(attrs: Dict[str, Any]) -> Dict[str, Any]:
    clean = {}
    for k, v in attrs.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        else:
            clean[k] = json.dumps(v, ensure_ascii=False)
    return clean


def close_ring(coords: List[List[float]]) -> List[Tuple[float, float]]:
    pts = [(float(x), float(y)) for x, y, *rest in coords]
    if len(pts) >= 3 and pts[0] != pts[-1]:
        pts.append(pts[0])
    return pts


def ring_signed_area(ring: List[Tuple[float, float]]) -> float:
    area = 0.0
    for i in range(len(ring) - 1):
        x1, y1 = ring[i]
        x2, y2 = ring[i + 1]
        area += x1 * y2 - x2 * y1
    return area / 2.0


def esri_polygon_to_shapely(rings_raw: List[List[List[float]]]):
    """
    Convert Esri polygon rings to Shapely.
    Esri often uses ring orientation to distinguish outer rings and holes.
    This function is conservative and tries to produce valid polygons.
    """
    rings = []
    for r in rings_raw:
        ring = close_ring(r)
        if len(ring) >= 4:
            rings.append(ring)

    if not rings:
        return None

    # Use orientation as a heuristic.
    # If it fails, fallback to first ring as exterior.
    outer_candidates = []
    hole_candidates = []

    for ring in rings:
        area = ring_signed_area(ring)
        # Esri convention often has outer rings clockwise.
        # In lon/lat with this formula, clockwise tends to be negative.
        if area < 0:
            outer_candidates.append(ring)
        else:
            hole_candidates.append(ring)

    if not outer_candidates:
        outer_candidates = [rings[0]]
        hole_candidates = rings[1:]

    polygons = []

    for outer in outer_candidates:
        outer_poly = Polygon(outer)
        holes_for_outer = []

        for hole in hole_candidates:
            try:
                test_point = Point(hole[0])
                if outer_poly.contains(test_point):
                    holes_for_outer.append(hole)
            except Exception:
                pass

        try:
            poly = Polygon(outer, holes=holes_for_outer)
            poly = make_valid(poly)
            if not poly.is_empty:
                polygons.append(poly)
        except Exception:
            pass

    if not polygons:
        try:
            poly = Polygon(rings[0], holes=rings[1:])
            poly = make_valid(poly)
            return poly if not poly.is_empty else None
        except Exception:
            return None

    if len(polygons) == 1:
        return polygons[0]

    mp = MultiPolygon([p for p in polygons if p.geom_type == "Polygon"])
    mp = make_valid(mp)
    return mp if not mp.is_empty else None


def esri_geometry_to_shapely(geom: Optional[Dict[str, Any]], geometry_type: str):
    if geom is None:
        return None

    if "x" in geom and "y" in geom:
        return Point(float(geom["x"]), float(geom["y"]))

    if "paths" in geom:
        lines = []
        for path in geom["paths"]:
            coords = [(float(x), float(y)) for x, y, *rest in path]
            if len(coords) >= 2:
                try:
                    lines.append(LineString(coords))
                except Exception:
                    pass
        if not lines:
            return None
        if len(lines) == 1:
            return lines[0]
        return MultiLineString(lines)

    if "rings" in geom:
        return esri_polygon_to_shapely(geom["rings"])

    return None


def query_features_by_oids_json(layer_url: str, oid_batch: List[int]) -> Dict[str, Any]:
    """
    Query ArcGIS features as Esri JSON.
    This is more stable than f=geojson for these layers.
    """
    params = {
        "objectIds": ",".join(str(x) for x in oid_batch),
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "json",
    }
    return request_json(layer_url + "/query", params=params, method="POST", timeout=300)


def esri_json_to_gdf(data: Dict[str, Any], geometry_type: str) -> gpd.GeoDataFrame:
    rows = []
    geoms = []

    for feat in data.get("features", []):
        attrs = sanitize_attributes(feat.get("attributes", {}))
        geom = esri_geometry_to_shapely(feat.get("geometry"), geometry_type)
        rows.append(attrs)
        geoms.append(geom)

    if not rows:
        return gpd.GeoDataFrame(columns=["geometry"], geometry=[], crs="EPSG:4326")

    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(df, geometry=geoms, crs="EPSG:4326")
    return gdf


def parquet_chunk_valid(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        gdf = gpd.read_parquet(path)
        return len(gdf) > 0
    except Exception:
        return False


def download_chunk(
    layer_name: str,
    layer_url: str,
    geometry_type: str,
    oid_batch: List[int],
    chunk_index: int,
    raw_dir: Path,
    parquet_dir: Path,
) -> Dict[str, Any]:
    parquet_path = parquet_dir / f"{layer_name}__chunk_{chunk_index:05d}.parquet"
    raw_path = raw_dir / f"{layer_name}__chunk_{chunk_index:05d}.json"

    if RESUME and parquet_chunk_valid(parquet_path):
        gdf = gpd.read_parquet(parquet_path)
        return {
            "layer": layer_name,
            "chunk_index": chunk_index,
            "status": "exists_valid",
            "n_oids_requested": len(oid_batch),
            "n_rows": len(gdf),
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "raw_json_path": str(raw_path) if raw_path.exists() else None,
        }

    try:
        data = query_features_by_oids_json(layer_url, oid_batch)

        if SAVE_RAW_JSON:
            tmp_raw = raw_path.with_suffix(".json.part")
            tmp_raw.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
            tmp_raw.replace(raw_path)

        gdf = esri_json_to_gdf(data, geometry_type=geometry_type)

        if len(gdf) == 0:
            raise RuntimeError("ArcGIS returned 0 features for this chunk.")

        tmp_parquet = parquet_path.with_suffix(".parquet.part")
        gdf.to_parquet(tmp_parquet, index=False)
        tmp_parquet.replace(parquet_path)

        return {
            "layer": layer_name,
            "chunk_index": chunk_index,
            "status": "downloaded",
            "n_oids_requested": len(oid_batch),
            "n_rows": len(gdf),
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "raw_json_path": str(raw_path),
            "raw_json_size_mb": file_size_mb(raw_path),
        }

    except Exception as e:
        return {
            "layer": layer_name,
            "chunk_index": chunk_index,
            "status": "failed",
            "n_oids_requested": len(oid_batch),
            "n_rows": None,
            "parquet_path": str(parquet_path),
            "raw_json_path": str(raw_path),
            "error": repr(e),
            "traceback": traceback.format_exc()[:3000],
        }


def combine_small_layer(layer_name: str, layer_dir: Path, expected_count: int) -> Dict[str, Any]:
    if not CREATE_SINGLE_FILE_FOR_SMALL_LAYERS:
        return {"single_file_status": "disabled"}

    if expected_count > SINGLE_FILE_MAX_FEATURES:
        return {"single_file_status": f"skipped_large_layer_{expected_count}_features"}

    parquet_dir = layer_dir / "parquet_chunks_v2"
    parquet_files = sorted(parquet_dir.glob("*.parquet"))

    if not parquet_files:
        return {"single_file_status": "no_parquet_chunks"}

    try:
        gdfs = [gpd.read_parquet(p) for p in parquet_files]
        full = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs="EPSG:4326")

        out_parquet = layer_dir / f"{layer_name}_v2.parquet"
        out_geojson = layer_dir / f"{layer_name}_v2.geojson"
        out_gpkg = layer_dir / f"{layer_name}_v2.gpkg"

        full.to_parquet(out_parquet, index=False)
        full.to_file(out_geojson, driver="GeoJSON")
        full.to_file(out_gpkg, layer=layer_name, driver="GPKG")

        return {
            "single_file_status": "created",
            "single_n_rows": len(full),
            "single_parquet_path": str(out_parquet),
            "single_geojson_path": str(out_geojson),
            "single_gpkg_path": str(out_gpkg),
        }

    except Exception as e:
        return {
            "single_file_status": "failed",
            "single_file_error": repr(e),
        }


def save_fields(metadata: Dict[str, Any], fields_path: Path) -> None:
    fields = metadata.get("fields", [])
    if fields:
        pd.DataFrame(fields).to_csv(fields_path, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# 4. Layer downloader
# ------------------------------------------------------------

def download_layer_v2(layer_name: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    print("\n" + "=" * 120)
    print("LAYER:", layer_name)
    print(cfg["description"])
    print("=" * 120)

    layer_url = cfg["url"]
    layer_dir = NTAD_ROOT / cfg["out_subdir"]

    metadata_dir = layer_dir / "metadata_v2"
    raw_dir = layer_dir / "raw_esri_json_chunks_v2"
    parquet_dir = layer_dir / "parquet_chunks_v2"

    for d in [layer_dir, metadata_dir, raw_dir, parquet_dir]:
        d.mkdir(parents=True, exist_ok=True)

    metadata = get_metadata(layer_url)
    oid_field = get_oid_field(metadata)
    geometry_type = metadata.get("geometryType", "unknown")
    max_record_count = int(metadata.get("maxRecordCount") or DEFAULT_BATCH_SIZE)
    batch_size = min(DEFAULT_BATCH_SIZE, max_record_count)

    expected_count = get_count(layer_url)
    object_ids = get_object_ids(layer_url)

    metadata_path = metadata_dir / f"{layer_name}_metadata_v2.json"
    fields_path = metadata_dir / f"{layer_name}_fields_v2.csv"
    object_ids_path = metadata_dir / f"{layer_name}_object_ids_v2.json"
    source_info_path = metadata_dir / f"{layer_name}_source_info_v2.json"

    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    save_fields(metadata, fields_path)
    object_ids_path.write_text(json.dumps(object_ids), encoding="utf-8")

    source_info = {
        "layer": layer_name,
        "description": cfg["description"],
        "layer_url": layer_url,
        "oid_field": oid_field,
        "geometry_type": geometry_type,
        "max_record_count": max_record_count,
        "batch_size": batch_size,
        "expected_count": expected_count,
        "n_object_ids": len(object_ids),
        "crs_requested": "EPSG:4326",
    }
    source_info_path.write_text(json.dumps(source_info, indent=2, ensure_ascii=False), encoding="utf-8")

    print("URL:", layer_url)
    print("OID field:", oid_field)
    print("Geometry type:", geometry_type)
    print("Expected count:", expected_count)
    print("Object IDs:", len(object_ids))
    print("Batch size:", batch_size)

    # Tiny probe before full download.
    print("\nProbe first 3 object IDs...")
    probe_ids = object_ids[:3]
    probe_data = query_features_by_oids_json(layer_url, probe_ids)
    probe_gdf = esri_json_to_gdf(probe_data, geometry_type=geometry_type)
    print("Probe rows:", len(probe_gdf))
    print("Probe columns:", list(probe_gdf.columns)[:30])
    if len(probe_gdf) == 0:
        raise RuntimeError(f"Probe failed for {layer_name}: 0 rows returned.")

    batches = chunk_list(object_ids, batch_size)

    chunk_manifest_path = layer_dir / f"{layer_name}_chunk_manifest_v2.csv"
    chunk_records = []

    # Load previous manifest if exists.
    if RESUME and chunk_manifest_path.exists():
        try:
            previous = pd.read_csv(chunk_manifest_path)
            chunk_records = previous.to_dict("records")
            print(f"Loaded existing chunk manifest: {chunk_manifest_path}")
        except Exception:
            chunk_records = []

    completed_chunks = {
        int(r["chunk_index"])
        for r in chunk_records
        if r.get("status") in {"downloaded", "exists_valid"}
    }

    t0 = time.time()

    for chunk_index, oid_batch in enumerate(tqdm(batches, desc=f"Downloading {layer_name}"), start=1):
        if chunk_index in completed_chunks:
            continue

        rec = download_chunk(
            layer_name=layer_name,
            layer_url=layer_url,
            geometry_type=geometry_type,
            oid_batch=oid_batch,
            chunk_index=chunk_index,
            raw_dir=raw_dir,
            parquet_dir=parquet_dir,
        )

        # If failed, retry in smaller subchunks.
        if (
            rec["status"] == "failed"
            and RETRY_FAILED_CHUNKS_WITH_SMALLER_BATCH
            and len(oid_batch) > SMALL_RETRY_BATCH_SIZE
        ):
            print(f"[small retry] {layer_name} chunk {chunk_index} failed. Retrying with batch={SMALL_RETRY_BATCH_SIZE}")
            sub_batches = chunk_list(oid_batch, SMALL_RETRY_BATCH_SIZE)
            sub_success_rows = 0
            sub_failed = 0

            for sub_i, sub_ids in enumerate(sub_batches, start=1):
                sub_chunk_index = int(f"{chunk_index}{sub_i:03d}")
                sub_rec = download_chunk(
                    layer_name=layer_name,
                    layer_url=layer_url,
                    geometry_type=geometry_type,
                    oid_batch=sub_ids,
                    chunk_index=sub_chunk_index,
                    raw_dir=raw_dir,
                    parquet_dir=parquet_dir,
                )
                chunk_records.append(sub_rec)
                if sub_rec["status"] in {"downloaded", "exists_valid"}:
                    sub_success_rows += int(sub_rec.get("n_rows") or 0)
                else:
                    sub_failed += 1

            rec = {
                "layer": layer_name,
                "chunk_index": chunk_index,
                "status": "split_retry_done",
                "n_oids_requested": len(oid_batch),
                "n_rows": sub_success_rows,
                "sub_failed": sub_failed,
                "parquet_path": None,
                "raw_json_path": None,
            }

        chunk_records.append(rec)

        # Save manifest frequently.
        if chunk_index % 5 == 0 or chunk_index == len(batches):
            pd.DataFrame(chunk_records).to_csv(chunk_manifest_path, index=False, encoding="utf-8-sig")

        if rec["status"] == "failed":
            print("[FAILED CHUNK]", layer_name, chunk_index, rec.get("error"))

        time.sleep(0.03)

    chunk_manifest = pd.DataFrame(chunk_records)
    chunk_manifest.to_csv(chunk_manifest_path, index=False, encoding="utf-8-sig")

    # Summarize from actual parquet files, not only manifest.
    parquet_files = sorted(parquet_dir.glob("*.parquet"))
    actual_rows = 0

    for p in tqdm(parquet_files, desc=f"Counting rows {layer_name}"):
        try:
            actual_rows += len(gpd.read_parquet(p))
        except Exception:
            pass

    single_info = combine_small_layer(layer_name, layer_dir, expected_count=len(object_ids))

    elapsed = round(time.time() - t0, 2)

    summary = {
        "layer": layer_name,
        "description": cfg["description"],
        "status": "ok" if actual_rows == len(object_ids) else "check_count_mismatch",
        "layer_url": layer_url,
        "layer_dir": str(layer_dir),
        "metadata_path": str(metadata_path),
        "fields_path": str(fields_path),
        "object_ids_path": str(object_ids_path),
        "source_info_path": str(source_info_path),
        "chunk_manifest_path": str(chunk_manifest_path),
        "expected_count_arcgis": expected_count,
        "n_object_ids": len(object_ids),
        "n_parquet_chunks": len(parquet_files),
        "actual_rows_from_parquet": actual_rows,
        "geometry_type": geometry_type,
        "elapsed_seconds": elapsed,
        "raw_dir": str(raw_dir),
        "parquet_dir": str(parquet_dir),
        **single_info,
    }

    summary_path = layer_dir / f"{layer_name}_summary_v2.json"
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    log_event(summary)

    print("\nSUMMARY")
    for k, v in summary.items():
        if k not in {"layer_url"}:
            print(f"  {k}: {v}")

    return summary


# ------------------------------------------------------------
# 5. Run
# ------------------------------------------------------------

print("=" * 120)
print("FREIGHT-MNet NTAD / ArcGIS downloader v2")
print("DATA_ROOT:", DATA_ROOT)
print("NTAD_ROOT:", NTAD_ROOT)
print("GLOBAL_MANIFEST_PATH:", GLOBAL_MANIFEST_PATH)
print("=" * 120)

all_summaries = []

for layer_name, cfg in LAYERS.items():
    try:
        summary = download_layer_v2(layer_name, cfg)
        all_summaries.append(summary)
    except Exception as e:
        err = {
            "layer": layer_name,
            "description": cfg.get("description"),
            "status": "failed",
            "error": repr(e),
            "traceback": traceback.format_exc()[:8000],
        }
        all_summaries.append(err)
        log_event(err)
        print("\nFAILED LAYER:", layer_name)
        print(e)
        print(traceback.format_exc())

global_manifest = pd.DataFrame(all_summaries)
global_manifest.to_csv(GLOBAL_MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\n" + "=" * 120)
print("NTAD / ArcGIS DOWNLOAD v2 COMPLETE")
print("Manifest:", GLOBAL_MANIFEST_PATH)
print("Log:", GLOBAL_LOG_PATH)
print("=" * 120)

cols = [
    "layer",
    "status",
    "expected_count_arcgis",
    "n_object_ids",
    "actual_rows_from_parquet",
    "n_parquet_chunks",
    "single_file_status",
    "layer_dir",
]
cols = [c for c in cols if c in global_manifest.columns]
display(global_manifest[cols])

FREIGHT-MNet NTAD / ArcGIS downloader v2
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
NTAD_ROOT: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial
GLOBAL_MANIFEST_PATH: E:\NetworkOptimization\pythonProject1\Data\00_manifest\ntad_arcgis_download_manifest_v2.csv

LAYER: faf_regions
FAF5 Regions / FAF zone polygons
URL: https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/NTAD_Freight_Analysis_Framework_Regions/FeatureServer/0
OID field: OBJECTID
Geometry type: esriGeometryPolygon
Expected count: 132
Object IDs: 132
Batch size: 1000

Probe first 3 object IDs...
Probe rows: 3
Probe columns: ['OBJECTID', 'CFS17_NAME', 'GEOID', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'CFS07_NAME', 'CFS12_NAME', 'CFS17_NA_1', 'FAF_Zone', 'FAF_Zone_D', 'FAF_Zone_1', 'Shape__Area', 'Shape__Length', 'geometry']


[small retry] faf_regions chunk 1 failed. Retrying with batch=100


Counting rows faf_regions: 0it [00:00, ?it/s]


SUMMARY
  layer: faf_regions
  description: FAF5 Regions / FAF zone polygons
  status: check_count_mismatch
  layer_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions
  metadata_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\metadata_v2\faf_regions_metadata_v2.json
  fields_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\metadata_v2\faf_regions_fields_v2.csv
  object_ids_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\metadata_v2\faf_regions_object_ids_v2.json
  source_info_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\metadata_v2\faf_regions_source_info_v2.json
  chunk_manifest_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\faf_regions\faf_regions_chunk_manifest_v2.csv
  expected_count_arcgis: 132
  n_object_ids: 132
  n_parquet_chunks: 0
  actual_rows_from_parquet: 0
  geometry_type: esriGeometryPolygon
  

[small retry] faf_network_links chunk 1 failed. Retrying with batch=100
[small retry] faf_network_links chunk 2 failed. Retrying with batch=100
[small retry] faf_network_links chunk 3 failed. Retrying with batch=100
[small retry] faf_network_links chunk 4 failed. Retrying with batch=100
[small retry] faf_network_links chunk 5 failed. Retrying with batch=100
[small retry] faf_network_links chunk 6 failed. Retrying with batch=100
[small retry] faf_network_links chunk 7 failed. Retrying with batch=100
[small retry] faf_network_links chunk 8 failed. Retrying with batch=100
[small retry] faf_network_links chunk 9 failed. Retrying with batch=100
[small retry] faf_network_links chunk 10 failed. Retrying with batch=100
[small retry] faf_network_links chunk 11 failed. Retrying with batch=100
[small retry] faf_network_links chunk 12 failed. Retrying with batch=100
[small retry] faf_network_links chunk 13 failed. Retrying with batch=100
[small retry] faf_network_links chunk 14 failed. Retrying wi

...


SUMMARY
  layer: narn_nodes
  description: North American Rail Network Nodes / rail node topology
  status: check_count_mismatch
  layer_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes
  metadata_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes\metadata_v2\narn_nodes_metadata_v2.json
  fields_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes\metadata_v2\narn_nodes_fields_v2.csv
  object_ids_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes\metadata_v2\narn_nodes_object_ids_v2.json
  source_info_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes\metadata_v2\narn_nodes_source_info_v2.json
  chunk_manifest_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\narn_nodes\narn_nodes_chunk_manifest_v2.csv
  expected_count_arcgis: 250178
  n_object_ids: 250178
  n_parquet_chunks: 0
  actual_rows_from_parquet: 0
  geometry_type: esriGe

[small retry] intermodal_rail_tofc_cofc chunk 1 failed. Retrying with batch=100


Counting rows intermodal_rail_tofc_cofc: 0it [00:00, ?it/s]


SUMMARY
  layer: intermodal_rail_tofc_cofc
  description: Intermodal Freight Facilities Rail TOFC/COFC / truck-rail transfer terminals
  status: check_count_mismatch
  layer_dir: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc
  metadata_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\metadata_v2\intermodal_rail_tofc_cofc_metadata_v2.json
  fields_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\metadata_v2\intermodal_rail_tofc_cofc_fields_v2.csv
  object_ids_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\metadata_v2\intermodal_rail_tofc_cofc_object_ids_v2.json
  source_info_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\intermodal_rail_tofc_cofc\metadata_v2\intermodal_rail_tofc_cofc_source_info_v2.json
  chunk_manifest_path: E:\NetworkOptimization\pythonProject1\Data\03_ntad_geospatial\in

,layer,status,expected_count_arcgis,n_object_ids,actual_rows_from_parquet,n_parquet_chunks,single_file_status,layer_dir
0,faf_regions,check_count_mismatch,132,132,0,0,no_parquet_chunks,E:\NetworkOptimization\pythonProject1\Data\03_...
1,faf_network_links,check_count_mismatch,487394,487394,0,0,skipped_large_layer_487394_features,E:\NetworkOptimization\pythonProject1\Data\03_...
2,faf_network_nodes,check_count_mismatch,974788,974788,0,0,skipped_large_layer_974788_features,E:\NetworkOptimization\pythonProject1\Data\03_...
3,narn_lines,check_count_mismatch,302743,302743,0,0,skipped_large_layer_302743_features,E:\NetworkOptimization\pythonProject1\Data\03_...
4,narn_nodes,check_count_mismatch,250178,250178,0,0,skipped_large_layer_250178_features,E:\NetworkOptimization\pythonProject1\Data\03_...
5,intermodal_rail_tofc_cofc,check_count_mismatch,241,241,0,0,no_parquet_chunks,E:\NetworkOptimization\pythonProject1\Data\03_...


In [4]:
# ============================================================
# FREIGHT-MNet: Download STB Public Use Waybill Sample
#
# Scope:
#   3.4 STB Public Use Waybill Sample only
#
# Years:
#   2018–2024
#
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use
#
# What this cell downloads:
#   - Annual Public Use Waybill Sample ZIP files
#   - Annual Reference Guide PDF files
#   - Creation / sampling procedure documents if found
#
# Output structure:
#   Data
#   ├── 00_manifest
#   │   ├── stb_waybill_download_manifest.csv
#   │   ├── stb_waybill_download_log.jsonl
#   │   └── stb_waybill_scraped_links_2018_2024.csv
#   │
#   └── 04_stb
#       └── waybill_public_use
#           ├── 2018
#           │   ├── sample_zip
#           │   ├── reference_guide
#           │   ├── extracted
#           │   └── preview
#           ├── ...
#           └── 2024
#
# This code is resumable:
#   If a file already exists and looks valid, it skips it.
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote
import os
import re
import json
import time
import zipfile
import hashlib
import traceback

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
WAYBILL_ROOT = DATA_ROOT / "04_stb" / "waybill_public_use"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
WAYBILL_ROOT.mkdir(parents=True, exist_ok=True)

STB_WAYBILL_PAGE = "https://www.stb.gov/reports-data/waybill/"
YEARS = list(range(2018, 2025))

MANIFEST_PATH = MANIFEST_ROOT / "stb_waybill_download_manifest.csv"
LOG_PATH = MANIFEST_ROOT / "stb_waybill_download_log.jsonl"
SCRAPED_LINKS_PATH = MANIFEST_ROOT / "stb_waybill_scraped_links_2018_2024.csv"

# Set True to extract ZIP files after download.
EXTRACT_ZIPS = True

# Set True if you want SHA256 checksums. Useful but slower.
COMPUTE_SHA256 = False

# If a local file exists and looks valid, skip it.
RESUME = True

# Timeout / retry settings.
CHUNK_SIZE = 1024 * 1024
SLEEP_BETWEEN_DOWNLOADS = 0.8

print("=" * 100)
print("STB Public Use Waybill downloader")
print("DATA_ROOT:", DATA_ROOT)
print("WAYBILL_ROOT:", WAYBILL_ROOT)
print("YEARS:", YEARS)
print("=" * 100)

# ------------------------------------------------------------
# 1. HTTP session
# ------------------------------------------------------------

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "FREIGHT-MNet academic research downloader "
            "(STB Public Use Waybill Samples, 2018-2024)"
        )
    }
)

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=1.5,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 3)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def safe_filename_from_url(url: str, fallback: str) -> str:
    parsed = urlparse(url)
    name = unquote(Path(parsed.path).name)
    if not name:
        name = fallback

    # Clean Windows-unfriendly characters.
    name = re.sub(r'[<>:"/\\|?*]+', "_", name)
    return name


def looks_like_valid_zip(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    if path.suffix.lower() != ".zip":
        return False
    try:
        with zipfile.ZipFile(path, "r") as zf:
            bad = zf.testzip()
            return bad is None and len(zf.namelist()) > 0
    except Exception:
        return False


def looks_like_valid_pdf(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        with path.open("rb") as f:
            header = f.read(5)
        return header == b"%PDF-"
    except Exception:
        return False


def looks_like_valid_file(path: Path, kind: str) -> bool:
    if kind == "sample_zip":
        return looks_like_valid_zip(path)
    if kind == "reference_guide":
        return looks_like_valid_pdf(path)
    if kind == "procedure_doc":
        # Procedure docs can be PDF or other documents.
        return path.exists() and path.stat().st_size > 1024
    return path.exists() and path.stat().st_size > 0


def download_file(url: str, out_path: Path, kind: str, label: str, overwrite: bool = False) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and out_path.exists() and not overwrite and looks_like_valid_file(out_path, kind):
        print(f"[skip existing valid] {label}: {out_path} ({file_size_mb(out_path)} MB)")
        log_event(
            {
                "status": "exists_valid",
                "kind": kind,
                "label": label,
                "url": url,
                "path": str(out_path),
                "size_mb": file_size_mb(out_path),
            }
        )
        return out_path

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    if part_path.exists():
        part_path.unlink()

    print("\n" + "-" * 100)
    print("[download]", label)
    print("URL:", url)
    print("OUT:", out_path)
    print("-" * 100)

    with session.get(url, stream=True, timeout=300) as r:
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code} while downloading {url}\n{r.text[:500]}")

        total = int(r.headers.get("content-length", 0))
        with part_path.open("wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=label,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    part_path.replace(out_path)

    # Validate downloaded file.
    if not looks_like_valid_file(out_path, kind):
        raise RuntimeError(f"Downloaded file does not look valid: {out_path}")

    log_event(
        {
            "status": "downloaded",
            "kind": kind,
            "label": label,
            "url": url,
            "path": str(out_path),
            "size_mb": file_size_mb(out_path),
        }
    )

    print(f"[saved] {out_path} ({file_size_mb(out_path)} MB)")
    return out_path


def extract_zip_if_needed(zip_path: Path, extract_dir: Path) -> dict:
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    marker = extract_dir / f".extracted_{zip_path.stem}.json"

    if RESUME and marker.exists():
        try:
            info = json.loads(marker.read_text(encoding="utf-8"))
            if info.get("status") == "ok":
                print(f"[skip extraction] {zip_path.name}")
                return info
        except Exception:
            pass

    print(f"[extract] {zip_path} -> {extract_dir}")

    extracted_files = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
        for name in zf.namelist():
            p = extract_dir / name
            if p.exists() and p.is_file():
                extracted_files.append(str(p))

    info = {
        "status": "ok",
        "zip_path": str(zip_path),
        "extract_dir": str(extract_dir),
        "n_extracted_files": len(extracted_files),
        "extracted_files": extracted_files,
    }

    marker.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")

    log_event(
        {
            "status": "extracted",
            "zip_path": str(zip_path),
            "extract_dir": str(extract_dir),
            "n_extracted_files": len(extracted_files),
        }
    )

    return info


def write_text_previews(extract_dir: Path, preview_dir: Path, year: int, max_lines: int = 20) -> list:
    extract_dir = Path(extract_dir)
    preview_dir = Path(preview_dir)
    preview_dir.mkdir(parents=True, exist_ok=True)

    preview_records = []

    candidate_files = []
    for pattern in ["*.txt", "*.TXT", "*.dat", "*.DAT", "*.csv", "*.CSV"]:
        candidate_files.extend(extract_dir.rglob(pattern))

    for p in sorted(set(candidate_files)):
        if not p.is_file():
            continue

        preview_path = preview_dir / f"{year}_{p.stem}_first{max_lines}_lines.txt"

        lines = []
        try:
            # Fixed-width public-use files are usually plain text.
            with p.open("r", encoding="utf-8", errors="replace") as f:
                for _ in range(max_lines):
                    line = f.readline()
                    if not line:
                        break
                    lines.append(line.rstrip("\n"))
        except Exception as e:
            preview_records.append(
                {
                    "year": year,
                    "source_file": str(p),
                    "preview_path": None,
                    "status": "failed",
                    "error": repr(e),
                }
            )
            continue

        preview_path.write_text("\n".join(lines), encoding="utf-8")

        preview_records.append(
            {
                "year": year,
                "source_file": str(p),
                "preview_path": str(preview_path),
                "status": "ok",
                "n_preview_lines": len(lines),
                "source_size_mb": file_size_mb(p),
            }
        )

    return preview_records


def classify_link(text: str, href: str) -> tuple[int | None, str | None]:
    """
    Return (year, kind).
    kind:
      - sample_zip
      - reference_guide
      - procedure_doc
      - None
    """
    combined = f"{text} {href}"

    year_match = re.search(r"(2018|2019|2020|2021|2022|2023|2024)", combined)
    year = int(year_match.group(1)) if year_match else None

    lower = combined.lower()

    if year in YEARS:
        if "public use waybill sample" in lower or "publicusewaybillsample" in lower:
            return year, "sample_zip"
        if "reference guide" in lower or "waybill-reference-guide" in lower:
            return year, "reference_guide"

    # Optional general procedure docs.
    if (
        "creation of the public use waybill sample" in lower
        or "procedure for sampling waybill records" in lower
        or "procedures for sampling waybill records" in lower
    ):
        return None, "procedure_doc"

    return year, None


def scrape_stb_waybill_links() -> pd.DataFrame:
    print("[scrape] STB Waybill page:", STB_WAYBILL_PAGE)
    r = session.get(STB_WAYBILL_PAGE, timeout=120)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    rows = []
    for a in soup.find_all("a", href=True):
        text = " ".join(a.get_text(" ", strip=True).split())
        href = urljoin(STB_WAYBILL_PAGE, a["href"])

        year, kind = classify_link(text, href)

        if kind in {"sample_zip", "reference_guide"} and year in YEARS:
            rows.append(
                {
                    "year": year,
                    "kind": kind,
                    "text": text,
                    "url": href,
                    "source": "scraped",
                }
            )

        elif kind == "procedure_doc":
            rows.append(
                {
                    "year": None,
                    "kind": kind,
                    "text": text,
                    "url": href,
                    "source": "scraped",
                }
            )

    df = pd.DataFrame(rows).drop_duplicates(subset=["year", "kind", "url"]).reset_index(drop=True)

    if not df.empty:
        df = df.sort_values(["kind", "year", "text"], na_position="last").reset_index(drop=True)

    return df


# Fallback URLs are only used if page scraping misses a year.
# The scraper is preferred because STB sometimes uses revised filenames.
FALLBACK_LINKS = []
for year in YEARS:
    FALLBACK_LINKS.append(
        {
            "year": year,
            "kind": "sample_zip",
            "text": f"{year} Public Use Waybill Sample",
            "url": f"https://www.stb.gov/wp-content/uploads/PublicUseWaybillSample{year}.zip",
            "source": "fallback_pattern",
        }
    )

    # Reference guide filenames are mostly regular, except 2021 revised/V2.
    if year == 2021:
        ref_url = "https://www.stb.gov/wp-content/uploads/2021-STB-Waybill-Reference-Guide-V2.pdf"
    else:
        ref_url = f"https://www.stb.gov/wp-content/uploads/{year}-STB-Waybill-Reference-Guide.pdf"

    FALLBACK_LINKS.append(
        {
            "year": year,
            "kind": "reference_guide",
            "text": f"Reference Guide – {year} Waybill Sample",
            "url": ref_url,
            "source": "fallback_pattern",
        }
    )


def complete_links_with_fallback(scraped: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure each year has one sample_zip and one reference_guide.
    Prefer scraped links. Fill missing with fallback patterns.
    """
    rows = []

    # Keep all scraped procedure docs.
    if not scraped.empty:
        rows.extend(scraped[scraped["kind"] == "procedure_doc"].to_dict("records"))

    for year in YEARS:
        for kind in ["sample_zip", "reference_guide"]:
            found = pd.DataFrame()
            if not scraped.empty:
                found = scraped[(scraped["year"] == year) & (scraped["kind"] == kind)]

            if len(found) > 0:
                # Prefer first scraped link.
                rows.append(found.iloc[0].to_dict())
            else:
                fallback = [x for x in FALLBACK_LINKS if x["year"] == year and x["kind"] == kind][0]
                rows.append(fallback)

    result = pd.DataFrame(rows).drop_duplicates(subset=["year", "kind", "url"]).reset_index(drop=True)
    result.to_csv(SCRAPED_LINKS_PATH, index=False, encoding="utf-8-sig")
    return result


def out_path_for_item(item: dict) -> Path:
    year = item.get("year")
    kind = item["kind"]
    url = item["url"]

    if kind == "procedure_doc":
        docs_dir = WAYBILL_ROOT / "_docs"
        docs_dir.mkdir(parents=True, exist_ok=True)
        filename = safe_filename_from_url(url, fallback=re.sub(r"\W+", "_", item["text"]) + ".pdf")
        return docs_dir / filename

    year = int(year)
    year_dir = WAYBILL_ROOT / str(year)

    if kind == "sample_zip":
        subdir = year_dir / "sample_zip"
        fallback = f"PublicUseWaybillSample{year}.zip"
    elif kind == "reference_guide":
        subdir = year_dir / "reference_guide"
        fallback = f"{year}-STB-Waybill-Reference-Guide.pdf"
    else:
        subdir = year_dir / "other"
        fallback = f"{year}_{kind}"

    subdir.mkdir(parents=True, exist_ok=True)
    filename = safe_filename_from_url(url, fallback=fallback)
    return subdir / filename


# ------------------------------------------------------------
# 3. Scrape links
# ------------------------------------------------------------

scraped_links = scrape_stb_waybill_links()

print("\nScraped links:")
display(scraped_links)

links = complete_links_with_fallback(scraped_links)

print("\nFinal links to download:")
display(links)

# Sanity check.
required = []
for year in YEARS:
    for kind in ["sample_zip", "reference_guide"]:
        ok = ((links["year"] == year) & (links["kind"] == kind)).any()
        required.append({"year": year, "kind": kind, "found": ok})

required_df = pd.DataFrame(required)
display(required_df)

if not required_df["found"].all():
    missing = required_df[required_df["found"] == False]
    raise RuntimeError(f"Missing required links:\n{missing}")

# ------------------------------------------------------------
# 4. Download all files
# ------------------------------------------------------------

manifest_rows = []
preview_rows = []

for item in links.to_dict("records"):
    kind = item["kind"]
    year = item.get("year")
    url = item["url"]
    text = item.get("text", "")
    source = item.get("source", "")

    out_path = out_path_for_item(item)

    label_year = str(int(year)) if pd.notna(year) else "docs"
    label = f"STB Waybill {label_year} | {kind}"

    record = {
        "year": int(year) if pd.notna(year) else None,
        "kind": kind,
        "text": text,
        "url": url,
        "source": source,
        "out_path": str(out_path),
    }

    try:
        downloaded_path = download_file(url, out_path, kind=kind, label=label)
        record["status"] = "downloaded_or_exists"
        record["size_mb"] = file_size_mb(downloaded_path)

        if COMPUTE_SHA256:
            record["sha256"] = sha256_file(downloaded_path)

        # For sample ZIPs, extract and preview.
        if kind == "sample_zip" and EXTRACT_ZIPS:
            year_int = int(year)
            extract_dir = WAYBILL_ROOT / str(year_int) / "extracted"
            preview_dir = WAYBILL_ROOT / str(year_int) / "preview"

            extract_info = extract_zip_if_needed(downloaded_path, extract_dir)

            record["extract_status"] = extract_info.get("status")
            record["n_extracted_files"] = extract_info.get("n_extracted_files")

            previews = write_text_previews(extract_dir, preview_dir, year_int, max_lines=20)
            preview_rows.extend(previews)

            record["n_preview_files"] = len(previews)

    except Exception as e:
        record["status"] = "failed"
        record["error"] = repr(e)
        record["traceback"] = traceback.format_exc()[:4000]
        print(f"[FAILED] {label}: {e}")

    manifest_rows.append(record)

    # Save progress after every file.
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

    if preview_rows:
        pd.DataFrame(preview_rows).to_csv(
            MANIFEST_ROOT / "stb_waybill_text_preview_manifest.csv",
            index=False,
            encoding="utf-8-sig",
        )

    time.sleep(SLEEP_BETWEEN_DOWNLOADS)

# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

if preview_rows:
    preview_manifest = pd.DataFrame(preview_rows)
    preview_manifest_path = MANIFEST_ROOT / "stb_waybill_text_preview_manifest.csv"
    preview_manifest.to_csv(preview_manifest_path, index=False, encoding="utf-8-sig")
else:
    preview_manifest_path = None

print("\n" + "=" * 100)
print("STB PUBLIC USE WAYBILL DOWNLOAD COMPLETE")
print("=" * 100)
print("WAYBILL_ROOT:", WAYBILL_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Log:", LOG_PATH)
print("Scraped links:", SCRAPED_LINKS_PATH)
print("Preview manifest:", preview_manifest_path)
print("=" * 100)

display_cols = [
    "year",
    "kind",
    "status",
    "source",
    "size_mb",
    "n_extracted_files",
    "n_preview_files",
    "out_path",
]
display_cols = [c for c in display_cols if c in manifest.columns]
display(manifest[display_cols])

# Fail loudly if any required annual sample/reference failed.
required_manifest = manifest[
    manifest["kind"].isin(["sample_zip", "reference_guide"])
].copy()

failed_required = required_manifest[required_manifest["status"] != "downloaded_or_exists"]

if len(failed_required) > 0:
    print("\nRequired files with problems:")
    display(failed_required[["year", "kind", "status", "url", "out_path", "error"]])
    raise RuntimeError("Some required STB Waybill files failed to download. See table above.")

print("\nAll required 2018–2024 Waybill sample ZIPs and Reference Guides are present.")

STB Public Use Waybill downloader
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
WAYBILL_ROOT: E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use
YEARS: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
[scrape] STB Waybill page: https://www.stb.gov/reports-data/waybill/

Scraped links:


,year,kind,text,url,source
0,NaN,procedure_doc,Creation of the Public Use Waybill Sample,https://www.stb.gov/wp-content/uploads/files/d...,scraped
1,NaN,procedure_doc,Procedure for Sampling Waybill Records by Comp...,https://www.stb.gov/wp-content/uploads/Procedu...,scraped
2,NaN,procedure_doc,Procedures for Sampling Waybill Records,https://www.stb.gov/wp-content/uploads/STB-Sta...,scraped
3,2018.0,reference_guide,Reference Guide – 2018 Waybill Sample,https://www.stb.gov/wp-content/uploads/2018-ST...,scraped
4,2019.0,reference_guide,Reference Guide – 2019 Waybill Sample,https://www.stb.gov/wp-content/uploads/2019-ST...,scraped
5,2020.0,reference_guide,Reference Guide – 2020 Waybill Sample,https://www.stb.gov/wp-content/uploads/2020-ST...,scraped
6,2021.0,reference_guide,Reference Guide – 2021 Waybill Sample (REVISED),https://www.stb.gov/wp-content/uploads/2021-ST...,scraped
7,2022.0,reference_guide,Reference Guide – 2022 Waybill Sample,https://www.stb.gov/wp-content/uploads/2022-ST...,scraped
8,2023.0,reference_guide,Reference Guide – 2023 Waybill Sample,https://www.stb.gov/wp-content/uploads/2023-ST...,scraped
9,2024.0,reference_guide,Reference Guide – 2024 Waybill Sample,https://www.stb.gov/wp-content/uploads/2024-ST...,scraped



Final links to download:


,year,kind,text,url,source
0,NaN,procedure_doc,Creation of the Public Use Waybill Sample,https://www.stb.gov/wp-content/uploads/files/d...,scraped
1,NaN,procedure_doc,Procedure for Sampling Waybill Records by Comp...,https://www.stb.gov/wp-content/uploads/Procedu...,scraped
2,NaN,procedure_doc,Procedures for Sampling Waybill Records,https://www.stb.gov/wp-content/uploads/STB-Sta...,scraped
3,2018.0,sample_zip,2018 Public Use Waybill Sample (REVISED),https://www.stb.gov/wp-content/uploads/PublicU...,scraped
4,2018.0,reference_guide,Reference Guide – 2018 Waybill Sample,https://www.stb.gov/wp-content/uploads/2018-ST...,scraped
5,2019.0,sample_zip,2019 Public Use Waybill Sample,https://www.stb.gov/wp-content/uploads/PublicU...,scraped
6,2019.0,reference_guide,Reference Guide – 2019 Waybill Sample,https://www.stb.gov/wp-content/uploads/2019-ST...,scraped
7,2020.0,sample_zip,2020 Public Use Waybill Sample,https://www.stb.gov/wp-content/uploads/PublicU...,scraped
8,2020.0,reference_guide,Reference Guide – 2020 Waybill Sample,https://www.stb.gov/wp-content/uploads/2020-ST...,scraped
9,2021.0,sample_zip,2021 Public Use Waybill Sample,https://www.stb.gov/wp-content/uploads/PublicU...,scraped


,year,kind,found
0,2018,sample_zip,True
1,2018,reference_guide,True
2,2019,sample_zip,True
3,2019,reference_guide,True
4,2020,sample_zip,True
5,2020,reference_guide,True
6,2021,sample_zip,True
7,2021,reference_guide,True
8,2022,sample_zip,True
9,2022,reference_guide,True


...

[saved] E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use\2024\sample_zip\PublicUseWaybillSample2024.zip (65.038 MB)
[extract] E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use\2024\sample_zip\PublicUseWaybillSample2024.zip -> E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use\2024\extracted

----------------------------------------------------------------------------------------------------
[download] STB Waybill 2024 | reference_guide
URL: https://www.stb.gov/wp-content/uploads/2024-STB-Waybill-Reference-Guide.pdf
OUT: E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use\2024\reference_guide\2024-STB-Waybill-Reference-Guide.pdf
----------------------------------------------------------------------------------------------------


STB Waybill 2024 | reference_guide:   0%|          | 0.00/5.20M [00:00<?, ?B/s]

[saved] E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use\2024\reference_guide\2024-STB-Waybill-Reference-Guide.pdf (4.96 MB)

STB PUBLIC USE WAYBILL DOWNLOAD COMPLETE
WAYBILL_ROOT: E:\NetworkOptimization\pythonProject1\Data\04_stb\waybill_public_use
Manifest: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_waybill_download_manifest.csv
Log: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_waybill_download_log.jsonl
Scraped links: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_waybill_scraped_links_2018_2024.csv
Preview manifest: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_waybill_text_preview_manifest.csv


,year,kind,status,source,size_mb,n_extracted_files,n_preview_files,out_path
0,NaN,procedure_doc,downloaded_or_exists,scraped,0.009,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
1,NaN,procedure_doc,downloaded_or_exists,scraped,0.091,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
2,NaN,procedure_doc,downloaded_or_exists,scraped,0.137,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
3,2018.0,sample_zip,downloaded_or_exists,scraped,20.890,7.0,1.0,E:\NetworkOptimization\pythonProject1\Data\04_...
4,2018.0,reference_guide,downloaded_or_exists,scraped,5.141,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
5,2019.0,sample_zip,downloaded_or_exists,scraped,20.460,7.0,1.0,E:\NetworkOptimization\pythonProject1\Data\04_...
6,2019.0,reference_guide,downloaded_or_exists,scraped,4.126,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
7,2020.0,sample_zip,downloaded_or_exists,scraped,19.242,7.0,1.0,E:\NetworkOptimization\pythonProject1\Data\04_...
8,2020.0,reference_guide,downloaded_or_exists,scraped,5.236,NaN,NaN,E:\NetworkOptimization\pythonProject1\Data\04_...
9,2021.0,sample_zip,downloaded_or_exists,scraped,62.273,7.0,1.0,E:\NetworkOptimization\pythonProject1\Data\04_...



All required 2018–2024 Waybill sample ZIPs and Reference Guides are present.


In [5]:
# ============================================================
# FREIGHT-MNet: Download STB Rail Service Performance Data
#
# Scope:
#   3.5 STB Rail Service Performance Data only
#
# Source:
#   https://www.stb.gov/reports-data/rail-service-data/
#
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance
#
# What this downloads:
#   1. Weekly rail service XLSX files, 2020–current page latest year
#      - All Class 1 Railroads consolidated files
#      - Individual carrier files: BNSF, CN, CP, CPKC, CSX, CTCO, KCS, NS, UP
#   2. STB-1145 CSV time-series files if present
#   3. Carrier methodology documents if present
#   4. Full link inventory and download manifest
#
# Resume behavior:
#   - If a file already exists and looks valid, it is skipped.
#   - Manifest is saved after every file.
#   - If interrupted, rerun the same cell.
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote
import os
import re
import json
import time
import hashlib
import zipfile
import traceback
from datetime import datetime, date

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from requests.utils import requote_uri

# Optional: only used for light workbook metadata.
try:
    import openpyxl
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
STB_RAIL_ROOT = DATA_ROOT / "04_stb" / "rail_service_performance"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

STB_PAGE_URL = "https://www.stb.gov/reports-data/rail-service-data/"

# Core research window is 2020-2024.
# We also download 2025-2026 if present because STB service data are current weekly data.
TARGET_YEARS = list(range(2020, datetime.now().year + 1))

# If you only want the core model period, uncomment this:
# TARGET_YEARS = list(range(2020, 2025))

DOWNLOAD_WEEKLY_XLSX = True
DOWNLOAD_WEEKLY_PDF = False       # Usually not needed for 2020+; set True if you want PDF copies too.
DOWNLOAD_STB1145_CSV = True       # Manifest/OETA/local-service time series CSV links, if present.
DOWNLOAD_METHODOLOGY_DOCS = True  # Carrier methodology docs.

# Keep existing valid files and skip them.
RESUME = True

# Compute SHA256 for downloaded files. Useful but slower.
COMPUTE_SHA256 = False

# Validate xlsx by checking ZIP structure. This is fast enough.
VALIDATE_XLSX = True

# Only collect workbook sheet names for first N XLSX files to avoid slow metadata scanning.
COLLECT_SHEET_NAMES_FOR_FIRST_N = 50

# Download pacing.
CHUNK_SIZE = 1024 * 1024
SLEEP_BETWEEN_DOWNLOADS = 0.20

# Paths.
STB_RAIL_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

RAW_WEEKLY_ROOT = STB_RAIL_ROOT / "weekly_service_files"
STB1145_ROOT = STB_RAIL_ROOT / "stb_1145_csv"
DOCS_ROOT = STB_RAIL_ROOT / "methodology_docs"
METADATA_ROOT = STB_RAIL_ROOT / "_metadata"

for d in [RAW_WEEKLY_ROOT, STB1145_ROOT, DOCS_ROOT, METADATA_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

LINK_INVENTORY_PATH = METADATA_ROOT / "stb_rail_service_link_inventory.csv"
DOWNLOAD_MANIFEST_PATH = MANIFEST_ROOT / "stb_rail_service_download_manifest.csv"
DOWNLOAD_LOG_PATH = MANIFEST_ROOT / "stb_rail_service_download_log.jsonl"
SHEET_PREVIEW_PATH = METADATA_ROOT / "stb_rail_service_xlsx_sheet_preview.csv"
PAGE_HTML_PATH = METADATA_ROOT / "stb_rail_service_page_snapshot.html"

print("=" * 100)
print("STB Rail Service Performance downloader")
print("DATA_ROOT:", DATA_ROOT)
print("STB_RAIL_ROOT:", STB_RAIL_ROOT)
print("TARGET_YEARS:", TARGET_YEARS)
print("HAS_OPENPYXL:", HAS_OPENPYXL)
print("=" * 100)

# ------------------------------------------------------------
# 1. HTTP session with retry
# ------------------------------------------------------------

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "FREIGHT-MNet academic research downloader "
            "(STB Rail Service Performance Data)"
        )
    }
)

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=1.5,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with DOWNLOAD_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 4)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def clean_windows_filename(name: str) -> str:
    name = unquote(name)
    name = re.sub(r'[<>:"/\\|?*]+', "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:180]


def short_hash(text: str, n: int = 10) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:n]


def parse_date_from_text_or_url(text: str, url: str):
    """
    Parse dates like:
      05-13-26.xlsx
      2026-05-13
    Returns: datetime.date or None
    """
    combined = f"{text} {url}"

    # Pattern 1: MM-DD-YY
    m = re.search(r"\b(\d{2})-(\d{2})-(\d{2})\b", combined)
    if m:
        mm, dd, yy = map(int, m.groups())
        yyyy = 2000 + yy if yy <= 69 else 1900 + yy
        try:
            return date(yyyy, mm, dd)
        except Exception:
            pass

    # Pattern 2: YYYY-MM-DD
    m = re.search(r"\b(20\d{2})-(\d{2})-(\d{2})\b", combined)
    if m:
        yyyy, mm, dd = map(int, m.groups())
        try:
            return date(yyyy, mm, dd)
        except Exception:
            pass

    return None


def get_file_ext(url: str, text: str = "") -> str:
    parsed = urlparse(url)
    base = unquote(Path(parsed.path).name).lower()
    m = re.search(r"\.(xlsx|xls|csv|pdf)(?:$|\?)", base)
    if m:
        return "." + m.group(1)

    m = re.search(r"\.(xlsx|xls|csv|pdf)\b", text.lower())
    if m:
        return "." + m.group(1)

    return ""


def guess_carrier_from_url(url: str, text: str = "", context: str = "") -> str:
    """
    Guess source group from URL path / anchor text / local context.
    STB URLs often contain folders like:
      /rsir/All Class 1 Railroads/
      /rsir/BNSF/
      /rsir/NS/
    """
    combined = unquote(f"{url} {text} {context}").lower()

    patterns = [
        ("all_class_1_railroads", [
            "all class 1 railroads",
            "all class i railroads",
            "consolidated data",
            "/all class 1 railroads/",
            "/all%20class%201%20railroads/",
        ]),
        ("bnsf", ["bnsf", "/bn/", "bn methodology"]),
        ("cn", ["canadian national", "/cn/", " cn "]),
        ("cp", ["canadian pacific", "/cp/", " cp "]),
        ("cpkc", ["cpkc"]),
        ("csx", ["csx", "/csx/"]),
        ("ctco", ["ctco", "chicago transportation coordination office"]),
        ("kcs", ["kansas city southern", "/kcs/", "kcsx"]),
        ("ns", ["norfolk southern", "/ns/", " ns "]),
        ("up", ["union pacific", "/up/", "uprr"]),
        ("stb_1145_all", ["stb-1145", "1145"]),
    ]

    for carrier, pats in patterns:
        for p in pats:
            if p in combined:
                return carrier

    return "unknown"


def looks_like_valid_xlsx(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    # XLSX is a ZIP container.
    return zipfile.is_zipfile(path)


def looks_like_valid_xls(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        with path.open("rb") as f:
            header = f.read(8)
        # OLE CFB header for old .xls:
        return header == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
    except Exception:
        return False


def looks_like_valid_pdf(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        with path.open("rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False


def looks_like_valid_csv(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        pd.read_csv(path, nrows=3)
        return True
    except Exception:
        # Some CSVs may have odd encodings; still accept if nonempty.
        return path.stat().st_size > 100


def looks_like_valid_file(path: Path, ext: str) -> bool:
    ext = ext.lower()
    if ext == ".xlsx":
        return looks_like_valid_xlsx(path) if VALIDATE_XLSX else path.exists() and path.stat().st_size > 0
    if ext == ".xls":
        return looks_like_valid_xls(path)
    if ext == ".pdf":
        return looks_like_valid_pdf(path)
    if ext == ".csv":
        return looks_like_valid_csv(path)
    return path.exists() and path.stat().st_size > 0


def download_file(url: str, out_path: Path, ext: str, label: str, overwrite: bool = False) -> Path:
    url = requote_uri(url)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and out_path.exists() and not overwrite and looks_like_valid_file(out_path, ext):
        print(f"[skip existing valid] {label}: {out_path.name} ({file_size_mb(out_path)} MB)")
        log_event({
            "status": "exists_valid",
            "url": url,
            "path": str(out_path),
            "size_mb": file_size_mb(out_path),
            "label": label,
        })
        return out_path

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    if part_path.exists():
        part_path.unlink()

    print("\n" + "-" * 100)
    print("[download]", label)
    print("URL:", url)
    print("OUT:", out_path)
    print("-" * 100)

    with session.get(url, stream=True, timeout=300) as r:
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code} while downloading {url}\n{r.text[:500]}")

        total = int(r.headers.get("content-length", 0))
        with part_path.open("wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=label[:60],
        ) as pbar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    part_path.replace(out_path)

    if not looks_like_valid_file(out_path, ext):
        raise RuntimeError(f"Downloaded file does not look valid: {out_path}")

    log_event({
        "status": "downloaded",
        "url": url,
        "path": str(out_path),
        "size_mb": file_size_mb(out_path),
        "label": label,
    })

    return out_path


def collect_xlsx_sheet_names(path: Path) -> dict:
    if not HAS_OPENPYXL:
        return {"sheet_status": "openpyxl_not_available"}

    try:
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
        sheets = wb.sheetnames
        wb.close()
        return {
            "sheet_status": "ok",
            "n_sheets": len(sheets),
            "sheet_names": "|".join(sheets),
        }
    except Exception as e:
        return {
            "sheet_status": "failed",
            "sheet_error": repr(e),
        }


# ------------------------------------------------------------
# 3. Scrape STB page
# ------------------------------------------------------------

print("[scrape]", STB_PAGE_URL)
r = session.get(STB_PAGE_URL, timeout=180)
r.raise_for_status()
html = r.text
PAGE_HTML_PATH.write_text(html, encoding="utf-8")

soup = BeautifulSoup(html, "html.parser")

# Gather all candidate links.
candidate_rows = []

for idx, a in enumerate(soup.find_all("a", href=True), start=1):
    raw_href = a["href"]
    url = urljoin(STB_PAGE_URL, raw_href)
    text = " ".join(a.get_text(" ", strip=True).split())
    ext = get_file_ext(url, text)

    if ext not in {".xlsx", ".xls", ".pdf", ".csv"}:
        continue

    # Context from nearby HTML to help classify.
    parent_text = ""
    try:
        parent_text = " ".join(a.parent.get_text(" ", strip=True).split())[:500]
    except Exception:
        parent_text = ""

    report_date = parse_date_from_text_or_url(text, url)
    year = report_date.year if report_date is not None else None

    carrier_guess = guess_carrier_from_url(url, text=text, context=parent_text)

    # Determine dataset group.
    lower_combined = unquote(f"{url} {text} {parent_text}").lower()

    if report_date is not None and ext in {".xlsx", ".xls", ".pdf"}:
        dataset_group = "weekly_service_report"
    elif ext == ".csv" and ("1145" in lower_combined or "stb-1145" in lower_combined):
        dataset_group = "stb_1145_csv"
    elif "methodology" in lower_combined or "methodologies" in lower_combined:
        dataset_group = "methodology_doc"
    else:
        # Keep other file links in inventory, but do not download unless explicitly selected.
        dataset_group = "other_file_link"

    candidate_rows.append({
        "link_seq": idx,
        "dataset_group": dataset_group,
        "carrier_guess": carrier_guess,
        "report_date": report_date.isoformat() if report_date else None,
        "year": year,
        "ext": ext,
        "anchor_text": text,
        "url": url,
        "parent_text": parent_text,
    })

link_inventory = pd.DataFrame(candidate_rows)

if link_inventory.empty:
    raise RuntimeError("No STB rail service file links were found on the page.")

# Filter what to download.
to_download_parts = []

if DOWNLOAD_WEEKLY_XLSX:
    weekly_xlsx = link_inventory[
        (link_inventory["dataset_group"] == "weekly_service_report")
        & (link_inventory["ext"].isin([".xlsx", ".xls"]))
        & (link_inventory["year"].isin(TARGET_YEARS))
    ].copy()
    to_download_parts.append(weekly_xlsx)

if DOWNLOAD_WEEKLY_PDF:
    weekly_pdf = link_inventory[
        (link_inventory["dataset_group"] == "weekly_service_report")
        & (link_inventory["ext"] == ".pdf")
        & (link_inventory["year"].isin(TARGET_YEARS))
    ].copy()
    to_download_parts.append(weekly_pdf)

if DOWNLOAD_STB1145_CSV:
    csv_1145 = link_inventory[
        (link_inventory["dataset_group"] == "stb_1145_csv")
        & (link_inventory["ext"] == ".csv")
    ].copy()
    to_download_parts.append(csv_1145)

if DOWNLOAD_METHODOLOGY_DOCS:
    docs = link_inventory[
        (link_inventory["dataset_group"].isin(["methodology_doc"]))
    ].copy()
    to_download_parts.append(docs)

if len(to_download_parts) == 0:
    raise RuntimeError("No selected download categories are enabled.")

download_inventory = pd.concat(to_download_parts, ignore_index=True).drop_duplicates(
    subset=["dataset_group", "url"]
).reset_index(drop=True)

# Save full inventory and selected inventory.
link_inventory.to_csv(LINK_INVENTORY_PATH, index=False, encoding="utf-8-sig")
selected_path = METADATA_ROOT / "stb_rail_service_selected_download_links.csv"
download_inventory.to_csv(selected_path, index=False, encoding="utf-8-sig")

print("\nFull link inventory:", LINK_INVENTORY_PATH)
print("Selected download links:", selected_path)
print("\nAll candidate link counts:")
display(link_inventory.groupby(["dataset_group", "ext"], dropna=False).size().reset_index(name="n_links"))

print("\nSelected download counts:")
display(download_inventory.groupby(["dataset_group", "carrier_guess", "year", "ext"], dropna=False).size().reset_index(name="n_links").head(50))

print("\nSelected total files:", len(download_inventory))

# ------------------------------------------------------------
# 4. Build output paths
# ------------------------------------------------------------

def output_path_for_row(row: pd.Series) -> Path:
    url = row["url"]
    ext = row["ext"]
    dataset_group = row["dataset_group"]
    carrier = row.get("carrier_guess") or "unknown"
    year = row.get("year")
    report_date = row.get("report_date")

    base = clean_windows_filename(Path(urlparse(url).path).name)
    if not base:
        base = f"stb_file_{row['link_seq']}{ext}"

    h = short_hash(url, 10)

    if dataset_group == "weekly_service_report":
        year_str = str(int(year)) if pd.notna(year) else "unknown_year"
        date_str = str(report_date) if pd.notna(report_date) and report_date else "unknown_date"
        subdir = RAW_WEEKLY_ROOT / carrier / year_str / ext.replace(".", "")
        out_name = f"{date_str}_{h}_{base}"
        return subdir / out_name

    if dataset_group == "stb_1145_csv":
        subdir = STB1145_ROOT / carrier
        out_name = f"{h}_{base}"
        return subdir / out_name

    if dataset_group == "methodology_doc":
        subdir = DOCS_ROOT / carrier
        out_name = f"{h}_{base}"
        return subdir / out_name

    subdir = STB_RAIL_ROOT / "other_file_links"
    out_name = f"{h}_{base}"
    return subdir / out_name


download_inventory["out_path"] = download_inventory.apply(lambda r: str(output_path_for_row(r)), axis=1)

# ------------------------------------------------------------
# 5. Download selected files
# ------------------------------------------------------------

manifest_rows = []
sheet_preview_rows = []
xlsx_sheet_counter = 0

# Load previous manifest if exists, so final output includes past runs.
if RESUME and DOWNLOAD_MANIFEST_PATH.exists():
    try:
        old_manifest = pd.read_csv(DOWNLOAD_MANIFEST_PATH)
        manifest_rows = old_manifest.to_dict("records")
        print(f"Loaded existing manifest with {len(manifest_rows)} rows.")
    except Exception:
        manifest_rows = []

# Use URL+out_path to avoid duplicate manifest rows during reruns.
existing_keys = {
    (str(r.get("url")), str(r.get("out_path")))
    for r in manifest_rows
    if r.get("status") in {"downloaded", "exists_valid", "downloaded_or_exists"}
}

for i, row in tqdm(list(download_inventory.iterrows()), total=len(download_inventory), desc="Downloading STB rail service files"):
    row_dict = row.to_dict()
    url = row_dict["url"]
    ext = row_dict["ext"]
    out_path = Path(row_dict["out_path"])

    key = (str(url), str(out_path))
    if RESUME and key in existing_keys and looks_like_valid_file(out_path, ext):
        continue

    label = f"{row_dict['dataset_group']} | {row_dict.get('carrier_guess')} | {row_dict.get('report_date') or ''} | {out_path.name}"

    rec = dict(row_dict)
    rec["out_path"] = str(out_path)

    try:
        downloaded = download_file(url, out_path, ext=ext, label=label)
        rec["status"] = "downloaded_or_exists"
        rec["size_mb"] = file_size_mb(downloaded)

        if COMPUTE_SHA256:
            rec["sha256"] = sha256_file(downloaded)

        if ext == ".xlsx" and xlsx_sheet_counter < COLLECT_SHEET_NAMES_FOR_FIRST_N:
            sheet_info = collect_xlsx_sheet_names(downloaded)
            sheet_preview_rows.append({
                "out_path": str(downloaded),
                "carrier_guess": rec.get("carrier_guess"),
                "report_date": rec.get("report_date"),
                "year": rec.get("year"),
                **sheet_info,
            })
            xlsx_sheet_counter += 1

    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = repr(e)
        rec["traceback"] = traceback.format_exc()[:4000]
        print(f"\n[FAILED] {label}")
        print(e)

    manifest_rows.append(rec)

    # Save progress after every file.
    pd.DataFrame(manifest_rows).to_csv(DOWNLOAD_MANIFEST_PATH, index=False, encoding="utf-8-sig")

    if sheet_preview_rows:
        pd.DataFrame(sheet_preview_rows).to_csv(SHEET_PREVIEW_PATH, index=False, encoding="utf-8-sig")

    time.sleep(SLEEP_BETWEEN_DOWNLOADS)

# ------------------------------------------------------------
# 6. Final manifest and summary
# ------------------------------------------------------------

manifest = pd.DataFrame(manifest_rows)

# Keep latest row for each URL/path pair.
if not manifest.empty:
    manifest = manifest.drop_duplicates(subset=["url", "out_path"], keep="last").reset_index(drop=True)

manifest.to_csv(DOWNLOAD_MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\n" + "=" * 100)
print("STB RAIL SERVICE PERFORMANCE DOWNLOAD COMPLETE")
print("=" * 100)
print("Root:", STB_RAIL_ROOT)
print("Full link inventory:", LINK_INVENTORY_PATH)
print("Selected links:", selected_path)
print("Download manifest:", DOWNLOAD_MANIFEST_PATH)
print("Download log:", DOWNLOAD_LOG_PATH)
print("Sheet preview:", SHEET_PREVIEW_PATH if SHEET_PREVIEW_PATH.exists() else "not created")
print("=" * 100)

summary_cols = [
    "dataset_group",
    "carrier_guess",
    "year",
    "ext",
    "status",
]
if not manifest.empty:
    summary = manifest.groupby(summary_cols, dropna=False).size().reset_index(name="n_files")
    display(summary)

    failed = manifest[manifest["status"] == "failed"].copy()
    if len(failed) > 0:
        print("\nFailed files:")
        display(failed[["dataset_group", "carrier_guess", "year", "ext", "url", "out_path", "error"]].head(50))
    else:
        print("\nNo failed files recorded.")

    print("\nDownloaded / existing file count:", len(manifest[manifest["status"].isin(["downloaded_or_exists", "exists_valid", "downloaded"])]))
    print("Total size GB:", round(manifest["size_mb"].fillna(0).sum() / 1024, 3))

STB Rail Service Performance downloader
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
STB_RAIL_ROOT: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance
TARGET_YEARS: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
HAS_OPENPYXL: True
[scrape] https://www.stb.gov/reports-data/rail-service-data/

Full link inventory: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance\_metadata\stb_rail_service_link_inventory.csv
Selected download links: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance\_metadata\stb_rail_service_selected_download_links.csv

All candidate link counts:


,dataset_group,ext,n_links
0,methodology_doc,.pdf,8
1,other_file_link,.csv,1
2,other_file_link,.pdf,1
3,stb_1145_csv,.csv,7
4,weekly_service_report,.pdf,1535
5,weekly_service_report,.xls,439
6,weekly_service_report,.xlsx,4335



Selected download counts:


,dataset_group,carrier_guess,year,ext,n_links
0,methodology_doc,bnsf,NaN,.pdf,1
1,methodology_doc,cn,NaN,.pdf,1
2,methodology_doc,cp,NaN,.pdf,1
3,methodology_doc,cpkc,NaN,.pdf,1
4,methodology_doc,csx,NaN,.pdf,1
5,methodology_doc,kcs,NaN,.pdf,1
6,methodology_doc,ns,NaN,.pdf,1
7,methodology_doc,up,NaN,.pdf,1
8,stb_1145_csv,bnsf,NaN,.csv,1
9,stb_1145_csv,cpkc,NaN,.csv,1



Selected total files: 2625


...


STB RAIL SERVICE PERFORMANCE DOWNLOAD COMPLETE
Root: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance
Full link inventory: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance\_metadata\stb_rail_service_link_inventory.csv
Selected links: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance\_metadata\stb_rail_service_selected_download_links.csv
Download manifest: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_rail_service_download_manifest.csv
Download log: E:\NetworkOptimization\pythonProject1\Data\00_manifest\stb_rail_service_download_log.jsonl
Sheet preview: E:\NetworkOptimization\pythonProject1\Data\04_stb\rail_service_performance\_metadata\stb_rail_service_xlsx_sheet_preview.csv


,dataset_group,carrier_guess,year,ext,status,n_files
0,methodology_doc,bnsf,NaN,.pdf,downloaded_or_exists,1
1,methodology_doc,cn,NaN,.pdf,downloaded_or_exists,1
2,methodology_doc,cp,NaN,.pdf,downloaded_or_exists,1
3,methodology_doc,cpkc,NaN,.pdf,downloaded_or_exists,1
4,methodology_doc,csx,NaN,.pdf,downloaded_or_exists,1
...,...,...,...,...,...,...
65,weekly_service_report,up,2022.0,.xlsx,downloaded_or_exists,52
66,weekly_service_report,up,2023.0,.xlsx,downloaded_or_exists,52
67,weekly_service_report,up,2024.0,.xlsx,downloaded_or_exists,52
68,weekly_service_report,up,2025.0,.xlsx,downloaded_or_exists,53



Failed files:


,dataset_group,carrier_guess,year,ext,url,out_path,error
1655,weekly_service_report,ctco,2020.0,.xlsx,https://www.stb.gov/wp-content/uploads/files/r...,E:\NetworkOptimization\pythonProject1\Data\04_...,RuntimeError('HTTP 404 while downloading https...



Downloaded / existing file count: 2624
Total size GB: 0.097


In [1]:
# ============================================================
# FREIGHT-MNet: Download NOAA Storm Events Database
#
# Scope:
#   3.6 NOAA Storm Events only
#
# Years:
#   2018–2024
#
# Source:
#   NOAA NCEI Storm Events bulk CSV directory
#
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events
#
# Downloads per year:
#   1. StormEvents_details-ftp_v1.0_dYYYY_c*.csv.gz
#   2. StormEvents_locations-ftp_v1.0_dYYYY_c*.csv.gz
#   3. StormEvents_fatalities-ftp_v1.0_dYYYY_c*.csv.gz
#
# Also downloads:
#   - README
#   - Storm-Data-Bulk-csv-Format.pdf
#   - Storm-Data-Export-Format.pdf
#
# Resume behavior:
#   - If a .csv.gz file already exists and passes gzip validation, it is skipped.
#   - If a .parquet file already exists and passes basic validation, conversion is skipped.
#   - Manifest is saved after every file.
#   - If interrupted, rerun the same cell.
# ============================================================

from pathlib import Path
from urllib.parse import urljoin
import re
import json
import time
import gzip
import hashlib
import traceback

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
NOAA_ROOT = DATA_ROOT / "05_noaa_storm_events"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

# Core model years. Keep this at 2018-2024 to match FMI / FAF supervised window.
YEARS = list(range(2018, 2025))

NOAA_INDEX_URL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"

TABLE_TYPES = {
    "details": {
        "subdir": "details",
        "pattern_prefix": "StormEvents_details-ftp_v1.0_d",
    },
    "locations": {
        "subdir": "locations",
        "pattern_prefix": "StormEvents_locations-ftp_v1.0_d",
    },
    "fatalities": {
        "subdir": "fatalities",
        "pattern_prefix": "StormEvents_fatalities-ftp_v1.0_d",
    },
}

DOC_FILES = [
    "README",
    "Storm-Data-Bulk-csv-Format.pdf",
    "Storm-Data-Export-Format.pdf",
]

# Output folders.
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
NOAA_ROOT.mkdir(parents=True, exist_ok=True)

DIRS = {
    "details": NOAA_ROOT / "details",
    "locations": NOAA_ROOT / "locations",
    "fatalities": NOAA_ROOT / "fatalities",
    "docs": NOAA_ROOT / "docs",
    "metadata": NOAA_ROOT / "_metadata",
    "parquet": NOAA_ROOT / "parquet",
    "preview": NOAA_ROOT / "preview",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

for table_type in TABLE_TYPES:
    (DIRS["parquet"] / table_type).mkdir(parents=True, exist_ok=True)
    (DIRS["preview"] / table_type).mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = MANIFEST_ROOT / "noaa_storm_events_download_manifest.csv"
LOG_PATH = MANIFEST_ROOT / "noaa_storm_events_download_log.jsonl"
INDEX_SNAPSHOT_PATH = DIRS["metadata"] / "noaa_storm_events_index_snapshot.html"
LINK_INVENTORY_PATH = DIRS["metadata"] / "noaa_storm_events_link_inventory_2018_2024.csv"

# Behavior switches.
RESUME = True
CONVERT_TO_PARQUET = True
SAVE_PREVIEWS = True
COMPUTE_SHA256 = False

# Download settings.
CHUNK_SIZE = 1024 * 1024
SLEEP_BETWEEN_DOWNLOADS = 0.25

print("=" * 100)
print("NOAA Storm Events downloader")
print("DATA_ROOT:", DATA_ROOT)
print("NOAA_ROOT:", NOAA_ROOT)
print("YEARS:", YEARS)
print("HAS_POLARS:", HAS_POLARS)
print("=" * 100)

# ------------------------------------------------------------
# 1. HTTP session with retry
# ------------------------------------------------------------

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "FREIGHT-MNet academic research downloader "
            "(NOAA NCEI Storm Events bulk CSV files)"
        )
    }
)

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=1.5,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 4)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def looks_like_valid_gzip_csv(path: Path) -> bool:
    """
    Basic validation:
      - file exists and nonempty
      - gzip can open it
      - first line contains comma-separated header
    """
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False

    try:
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            header = f.readline()
        return "," in header and len(header.strip()) > 5
    except Exception:
        return False


def looks_like_valid_pdf(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        with path.open("rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False


def looks_like_valid_text(path: Path) -> bool:
    path = Path(path)
    return path.exists() and path.stat().st_size > 0


def looks_like_valid_parquet(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False

    try:
        if HAS_POLARS:
            n = pl.scan_parquet(str(path)).select(pl.len()).collect().item()
            return n >= 0
        else:
            sample = pd.read_parquet(path)
            return sample.shape[1] > 0
    except Exception:
        return False


def download_file(url: str, out_path: Path, kind: str, label: str, overwrite: bool = False) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    def valid_existing() -> bool:
        if kind == "gzip_csv":
            return looks_like_valid_gzip_csv(out_path)
        if kind == "pdf":
            return looks_like_valid_pdf(out_path)
        if kind == "text":
            return looks_like_valid_text(out_path)
        return out_path.exists() and out_path.stat().st_size > 0

    if RESUME and out_path.exists() and not overwrite and valid_existing():
        print(f"[skip existing valid] {label}: {out_path.name} ({file_size_mb(out_path)} MB)")
        log_event(
            {
                "status": "exists_valid",
                "kind": kind,
                "url": url,
                "path": str(out_path),
                "size_mb": file_size_mb(out_path),
                "label": label,
            }
        )
        return out_path

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    if part_path.exists():
        part_path.unlink()

    print("\n" + "-" * 100)
    print("[download]", label)
    print("URL:", url)
    print("OUT:", out_path)
    print("-" * 100)

    with session.get(url, stream=True, timeout=300) as r:
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code} while downloading {url}\n{r.text[:500]}")

        total = int(r.headers.get("content-length", 0))
        with part_path.open("wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=label[:70],
        ) as pbar:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    part_path.replace(out_path)

    if not valid_existing():
        raise RuntimeError(f"Downloaded file did not pass validation: {out_path}")

    log_event(
        {
            "status": "downloaded",
            "kind": kind,
            "url": url,
            "path": str(out_path),
            "size_mb": file_size_mb(out_path),
            "label": label,
        }
    )

    return out_path


def fetch_index_html() -> str:
    r = session.get(NOAA_INDEX_URL, timeout=120)
    r.raise_for_status()
    html = r.text
    INDEX_SNAPSHOT_PATH.write_text(html, encoding="utf-8")
    return html


def parse_noaa_index(html: str) -> pd.DataFrame:
    """
    Parse NOAA HTTP directory listing and return all StormEvents files.
    """
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        filename = href.split("/")[-1]

        if not filename.startswith("StormEvents_"):
            continue
        if not filename.endswith(".csv.gz"):
            continue

        m = re.match(
            r"StormEvents_(details|locations|fatalities)-ftp_v1\.0_d(\d{4})_c(\d{8})\.csv\.gz",
            filename,
        )

        if not m:
            continue

        table_type, year, creation_date = m.groups()
        year = int(year)
        url = urljoin(NOAA_INDEX_URL, filename)

        rows.append(
            {
                "table_type": table_type,
                "year": year,
                "creation_date": creation_date,
                "filename": filename,
                "url": url,
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError("No StormEvents CSV.GZ files were found in NOAA index.")

    df = df.sort_values(["table_type", "year", "creation_date"]).reset_index(drop=True)
    return df


def select_latest_files_for_years(inventory: pd.DataFrame, years: list[int]) -> pd.DataFrame:
    """
    NOAA may keep only one creation version per year, but this safely selects latest cYYYYMMDD.
    """
    subset = inventory[inventory["year"].isin(years)].copy()
    needed_rows = []

    for table_type in TABLE_TYPES:
        for year in years:
            candidates = subset[(subset["table_type"] == table_type) & (subset["year"] == year)].copy()
            if len(candidates) == 0:
                needed_rows.append(
                    {
                        "table_type": table_type,
                        "year": year,
                        "status": "missing_in_index",
                        "filename": None,
                        "url": None,
                    }
                )
                continue

            latest = candidates.sort_values("creation_date").iloc[-1].to_dict()
            latest["status"] = "selected"
            needed_rows.append(latest)

    selected = pd.DataFrame(needed_rows)
    return selected


def preview_gzip_csv(gz_path: Path, preview_path: Path, nrows: int = 20) -> dict:
    preview_path = Path(preview_path)
    preview_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        df = pd.read_csv(gz_path, compression="gzip", nrows=nrows, low_memory=False)
        df.to_csv(preview_path, index=False, encoding="utf-8-sig")

        schema_path = preview_path.with_name(preview_path.stem + "_schema.csv")
        schema = pd.DataFrame(
            {
                "column": df.columns,
                "preview_dtype": [str(t) for t in df.dtypes],
            }
        )
        schema.to_csv(schema_path, index=False, encoding="utf-8-sig")

        return {
            "preview_status": "ok",
            "preview_path": str(preview_path),
            "schema_path": str(schema_path),
            "n_preview_rows": len(df),
            "n_preview_columns": len(df.columns),
            "columns": "|".join(df.columns),
        }

    except Exception as e:
        return {
            "preview_status": "failed",
            "preview_error": repr(e),
        }


def convert_gzip_csv_to_parquet(gz_path: Path, parquet_path: Path, table_type: str, year: int) -> dict:
    parquet_path = Path(parquet_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and looks_like_valid_parquet(parquet_path):
        print(f"[skip existing parquet] {parquet_path.name} ({file_size_mb(parquet_path)} MB)")

        try:
            if HAS_POLARS:
                n_rows = pl.scan_parquet(str(parquet_path)).select(pl.len()).collect().item()
                columns = pl.scan_parquet(str(parquet_path)).collect_schema().names()
            else:
                tmp = pd.read_parquet(parquet_path)
                n_rows = len(tmp)
                columns = list(tmp.columns)
        except Exception:
            n_rows = None
            columns = []

        return {
            "parquet_status": "exists_valid",
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "parquet_rows": n_rows,
            "parquet_columns": "|".join(columns),
        }

    print("\n" + "-" * 100)
    print(f"[convert] NOAA {table_type} {year}: CSV.GZ -> Parquet")
    print("GZ:", gz_path)
    print("PARQUET:", parquet_path)
    print("-" * 100)

    try:
        if HAS_POLARS:
            df = pl.read_csv(
                str(gz_path),
                infer_schema_length=10000,
                ignore_errors=True,
                try_parse_dates=False,
            )

            n_rows = df.height
            columns = df.columns
            tmp_path = parquet_path.with_suffix(".parquet.part")
            df.write_parquet(str(tmp_path), compression="zstd")
            tmp_path.replace(parquet_path)
            del df

        else:
            # Fallback: pandas full read. For 2018-2024 this should still be manageable.
            df = pd.read_csv(gz_path, compression="gzip", low_memory=False)
            n_rows = len(df)
            columns = list(df.columns)
            tmp_path = parquet_path.with_suffix(".parquet.part")
            df.to_parquet(tmp_path, index=False)
            tmp_path.replace(parquet_path)
            del df

        return {
            "parquet_status": "converted",
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "parquet_rows": n_rows,
            "parquet_columns": "|".join(columns),
        }

    except Exception as e:
        return {
            "parquet_status": "failed",
            "parquet_path": str(parquet_path),
            "parquet_error": repr(e),
            "parquet_traceback": traceback.format_exc()[:3000],
        }


# ------------------------------------------------------------
# 3. Scrape NOAA index and select latest files
# ------------------------------------------------------------

html = fetch_index_html()
inventory = parse_noaa_index(html)
inventory.to_csv(DIRS["metadata"] / "noaa_storm_events_full_index_inventory.csv", index=False, encoding="utf-8-sig")

selected = select_latest_files_for_years(inventory, YEARS)
selected.to_csv(LINK_INVENTORY_PATH, index=False, encoding="utf-8-sig")

print("\nNOAA selected files:")
display(selected)

missing = selected[selected["status"] != "selected"]
if len(missing) > 0:
    print("\nMissing files in NOAA index:")
    display(missing)
    raise RuntimeError("Some required NOAA files were missing from the index. See table above.")

# ------------------------------------------------------------
# 4. Download documentation
# ------------------------------------------------------------

doc_manifest_rows = []

for doc_name in DOC_FILES:
    url = urljoin(NOAA_INDEX_URL, doc_name)
    out_path = DIRS["docs"] / doc_name
    kind = "pdf" if doc_name.lower().endswith(".pdf") else "text"

    rec = {
        "record_type": "doc",
        "filename": doc_name,
        "url": url,
        "out_path": str(out_path),
    }

    try:
        p = download_file(url, out_path, kind=kind, label=f"NOAA doc {doc_name}")
        rec["status"] = "downloaded_or_exists"
        rec["size_mb"] = file_size_mb(p)
        if COMPUTE_SHA256:
            rec["sha256"] = sha256_file(p)

    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = repr(e)
        rec["traceback"] = traceback.format_exc()[:3000]
        print(f"[doc failed] {doc_name}: {e}")

    doc_manifest_rows.append(rec)
    time.sleep(SLEEP_BETWEEN_DOWNLOADS)

# ------------------------------------------------------------
# 5. Download data files, preview, and convert to Parquet
# ------------------------------------------------------------

manifest_rows = []

# Load previous manifest if rerunning.
if RESUME and MANIFEST_PATH.exists():
    try:
        old = pd.read_csv(MANIFEST_PATH)
        # Keep previous docs and data rows; final drop_duplicates at end.
        manifest_rows = old.to_dict("records")
        print(f"Loaded previous manifest with {len(manifest_rows)} rows.")
    except Exception:
        manifest_rows = []

# Add docs to manifest.
manifest_rows.extend(doc_manifest_rows)

for _, row in tqdm(list(selected.iterrows()), total=len(selected), desc="Downloading NOAA Storm Events"):
    table_type = row["table_type"]
    year = int(row["year"])
    filename = row["filename"]
    url = row["url"]

    out_dir = DIRS[table_type]
    out_path = out_dir / filename

    preview_path = DIRS["preview"] / table_type / f"noaa_stormevents_{table_type}_{year}_preview_first20.csv"
    parquet_path = DIRS["parquet"] / table_type / f"noaa_stormevents_{table_type}_{year}.parquet"

    label = f"NOAA StormEvents {table_type} {year}"

    rec = {
        "record_type": "data",
        "table_type": table_type,
        "year": year,
        "creation_date": row["creation_date"],
        "filename": filename,
        "url": url,
        "out_path": str(out_path),
        "preview_path": str(preview_path),
        "parquet_path": str(parquet_path),
    }

    try:
        gz_path = download_file(url, out_path, kind="gzip_csv", label=label)
        rec["status"] = "downloaded_or_exists"
        rec["size_mb"] = file_size_mb(gz_path)

        if COMPUTE_SHA256:
            rec["sha256"] = sha256_file(gz_path)

        if SAVE_PREVIEWS:
            preview_info = preview_gzip_csv(gz_path, preview_path, nrows=20)
            rec.update(preview_info)

        if CONVERT_TO_PARQUET:
            parquet_info = convert_gzip_csv_to_parquet(gz_path, parquet_path, table_type, year)
            rec.update(parquet_info)

    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = repr(e)
        rec["traceback"] = traceback.format_exc()[:4000]
        print(f"\n[FAILED] {label}: {e}")

    manifest_rows.append(rec)

    # Save progress after each file.
    pd.DataFrame(manifest_rows).drop_duplicates(
        subset=["record_type", "table_type", "year", "filename", "out_path"],
        keep="last",
    ).to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

    time.sleep(SLEEP_BETWEEN_DOWNLOADS)

# ------------------------------------------------------------
# 6. Final manifest and summary
# ------------------------------------------------------------

manifest = pd.DataFrame(manifest_rows)

# Drop exact duplicated records from reruns.
dedup_cols = [c for c in ["record_type", "table_type", "year", "filename", "out_path"] if c in manifest.columns]
if dedup_cols:
    manifest = manifest.drop_duplicates(subset=dedup_cols, keep="last").reset_index(drop=True)

manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\n" + "=" * 100)
print("NOAA STORM EVENTS DOWNLOAD COMPLETE")
print("=" * 100)
print("NOAA_ROOT:", NOAA_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Log:", LOG_PATH)
print("Index snapshot:", INDEX_SNAPSHOT_PATH)
print("Link inventory:", LINK_INVENTORY_PATH)
print("=" * 100)

data_manifest = manifest[manifest["record_type"] == "data"].copy()
doc_manifest = manifest[manifest["record_type"] == "doc"].copy()

print("\nData files summary:")
if len(data_manifest) > 0:
    cols = [
        "table_type",
        "year",
        "status",
        "size_mb",
        "preview_status",
        "parquet_status",
        "parquet_rows",
        "out_path",
        "parquet_path",
    ]
    cols = [c for c in cols if c in data_manifest.columns]
    display(data_manifest[cols].sort_values(["table_type", "year"]))

print("\nDocs summary:")
if len(doc_manifest) > 0:
    cols = ["filename", "status", "size_mb", "out_path"]
    cols = [c for c in cols if c in doc_manifest.columns]
    display(doc_manifest[cols])

failed = manifest[manifest["status"] == "failed"].copy()
if len(failed) > 0:
    print("\nFAILED records:")
    display(failed)
    raise RuntimeError("Some NOAA files failed. Rerun the cell to resume.")
else:
    print("\nAll selected NOAA Storm Events files are present.")

NOAA Storm Events downloader
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
NOAA_ROOT: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events
YEARS: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
HAS_POLARS: False

NOAA selected files:


,table_type,year,creation_date,filename,url,status
0,details,2018,20260323,StormEvents_details-ftp_v1.0_d2018_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
1,details,2019,20260323,StormEvents_details-ftp_v1.0_d2019_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
2,details,2020,20260323,StormEvents_details-ftp_v1.0_d2020_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
3,details,2021,20260323,StormEvents_details-ftp_v1.0_d2021_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
4,details,2022,20260323,StormEvents_details-ftp_v1.0_d2022_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
5,details,2023,20260323,StormEvents_details-ftp_v1.0_d2023_c20260323.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
6,details,2024,20260421,StormEvents_details-ftp_v1.0_d2024_c20260421.c...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
7,locations,2018,20260323,StormEvents_locations-ftp_v1.0_d2018_c20260323...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
8,locations,2019,20260323,StormEvents_locations-ftp_v1.0_d2019_c20260323...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected
9,locations,2020,20260323,StormEvents_locations-ftp_v1.0_d2020_c20260323...,https://www.ncei.noaa.gov/pub/data/swdi/storme...,selected



----------------------------------------------------------------------------------------------------
[download] NOAA doc README
URL: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/README
OUT: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\docs\README
----------------------------------------------------------------------------------------------------


NOAA doc README:   0%|          | 0.00/2.01k [00:00<?, ?B/s]


----------------------------------------------------------------------------------------------------
[download] NOAA doc Storm-Data-Bulk-csv-Format.pdf
URL: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf
OUT: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\docs\Storm-Data-Bulk-csv-Format.pdf
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
[convert] NOAA fatalities 2024: CSV.GZ -> Parquet
GZ: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\fatalities\StormEvents_fatalities-ftp_v1.0_d2024_c20260421.csv.gz
PARQUET: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\parquet\fatalities\noaa_stormevents_fatalities_2024.parquet
----------------------------------------------------------------------------------------------------

NOAA STORM EVENTS DOWNLOAD COMPLETE
NOAA_ROOT: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events
Manifest: E:\NetworkOptimization\pythonProject1\Data\00_manifest\noaa_storm_events_download_manifest.csv
Log: E:\NetworkOptimization\pythonProject1\Data\00_manifest\noaa_storm_events_download_log.jsonl
Index snapshot: E:\NetworkOptimization\pythonProject1\Data\05_noaa_storm_events\_metadata\noaa_storm_events_index_snapshot.html
Link inventory: E:\NetworkOptimization

,table_type,year,status,size_mb,preview_status,parquet_status,parquet_rows,out_path,parquet_path
3,details,2018.0,downloaded_or_exists,9.1419,ok,converted,62699.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
4,details,2019.0,downloaded_or_exists,11.0651,ok,converted,67864.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
5,details,2020.0,downloaded_or_exists,9.9608,ok,converted,61281.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
6,details,2021.0,downloaded_or_exists,10.0746,ok,converted,61389.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
7,details,2022.0,downloaded_or_exists,11.2718,ok,converted,69887.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
8,details,2023.0,downloaded_or_exists,12.2910,ok,converted,75593.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
9,details,2024.0,downloaded_or_exists,12.1054,ok,converted,69801.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
17,fatalities,2018.0,downloaded_or_exists,0.0114,ok,converted,1072.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
18,fatalities,2019.0,downloaded_or_exists,0.0091,ok,converted,733.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...
19,fatalities,2020.0,downloaded_or_exists,0.0095,ok,converted,905.0,E:\NetworkOptimization\pythonProject1\Data\05_...,E:\NetworkOptimization\pythonProject1\Data\05_...



Docs summary:


,filename,status,size_mb,out_path
0,README,downloaded_or_exists,0.0019,E:\NetworkOptimization\pythonProject1\Data\05_...
1,Storm-Data-Bulk-csv-Format.pdf,downloaded_or_exists,0.3254,E:\NetworkOptimization\pythonProject1\Data\05_...
2,Storm-Data-Export-Format.pdf,downloaded_or_exists,0.1436,E:\NetworkOptimization\pythonProject1\Data\05_...



All selected NOAA Storm Events files are present.


In [2]:
# ============================================================
# FREIGHT-MNet: Download BTS Border Crossing Entry Data
#
# Scope:
#   3.7 BTS Border Crossing Entry Data only
#
# Source:
#   BTS / DOT Socrata dataset: keg4-3bc2
#
# Save root:
#   E:\NetworkOptimization\pythonProject1\Data\06_border_crossing
#
# What this cell produces:
#   1. Full raw CSV snapshot
#   2. Full raw Parquet snapshot
#   3. Resumable Socrata API chunk CSV files
#   4. Resumable Socrata API chunk Parquet files
#   5. Metadata JSON
#   6. Columns JSON + columns CSV
#   7. Preview CSV
#   8. Measure inventory
#   9. Core 2018–2024 freight subset:
#        truck / truck container / rail / train measures
#
# Resume behavior:
#   - Existing valid files are skipped.
#   - Existing valid chunks are skipped.
#   - If interrupted, rerun this same cell.
# ============================================================

from pathlib import Path
from urllib.parse import urljoin
import json
import time
import hashlib
import traceback
import math
import re

import requests
import pandas as pd
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
BORDER_ROOT = DATA_ROOT / "06_border_crossing"
MANIFEST_ROOT = DATA_ROOT / "00_manifest"

DATASET_ID = "keg4-3bc2"

# Try BTS domain first, then DOT transportation domain.
SOCRATA_DOMAINS = [
    "https://data.bts.gov",
    "https://data.transportation.gov",
]

# Main modeling window. Raw full dataset is downloaded; this window is used for processed subset.
CORE_YEARS = list(range(2018, 2025))

# Chunk size for Socrata API paging.
# Dataset is not huge, but chunking gives true resume behavior.
CHUNK_SIZE = 50_000

# Behavior switches.
RESUME = True
DOWNLOAD_FULL_SNAPSHOT = True
DOWNLOAD_CHUNKS = True
CONVERT_TO_PARQUET = True
SAVE_CORE_2018_2024_FREIGHT_SUBSET = True
COMPUTE_SHA256 = False

# Folders.
DIRS = {
    "metadata": BORDER_ROOT / "_metadata",
    "raw_full": BORDER_ROOT / "raw_full_snapshot",
    "chunks_csv": BORDER_ROOT / "chunks_csv",
    "chunks_parquet": BORDER_ROOT / "chunks_parquet",
    "parquet": BORDER_ROOT / "parquet",
    "preview": BORDER_ROOT / "preview",
    "processed": BORDER_ROOT / "processed",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = MANIFEST_ROOT / "bts_border_crossing_download_manifest.csv"
LOG_PATH = MANIFEST_ROOT / "bts_border_crossing_download_log.jsonl"

FULL_CSV_PATH = DIRS["raw_full"] / f"border_crossing_entry_data_{DATASET_ID}_full.csv"
FULL_PARQUET_PATH = DIRS["parquet"] / f"border_crossing_entry_data_{DATASET_ID}_full.parquet"

METADATA_JSON_PATH = DIRS["metadata"] / f"border_crossing_entry_data_{DATASET_ID}_metadata.json"
COLUMNS_JSON_PATH = DIRS["metadata"] / f"border_crossing_entry_data_{DATASET_ID}_columns.json"
COLUMNS_CSV_PATH = DIRS["metadata"] / f"border_crossing_entry_data_{DATASET_ID}_columns.csv"
SOURCE_INFO_PATH = DIRS["metadata"] / f"border_crossing_entry_data_{DATASET_ID}_source_info.json"

PREVIEW_PATH = DIRS["preview"] / f"border_crossing_entry_data_{DATASET_ID}_preview_first50.csv"
MEASURE_INVENTORY_PATH = DIRS["processed"] / "border_crossing_measure_inventory.csv"
CORE_FREIGHT_PARQUET_PATH = DIRS["processed"] / "border_crossing_freight_measures_2018_2024.parquet"
CORE_FREIGHT_CSV_PATH = DIRS["processed"] / "border_crossing_freight_measures_2018_2024.csv"
MONTHLY_GATEWAY_PIVOT_PATH = DIRS["processed"] / "border_crossing_gateway_monthly_pivot_2018_2024.parquet"

print("=" * 100)
print("BTS Border Crossing Entry Data downloader")
print("DATA_ROOT:", DATA_ROOT)
print("BORDER_ROOT:", BORDER_ROOT)
print("DATASET_ID:", DATASET_ID)
print("CORE_YEARS:", CORE_YEARS)
print("HAS_POLARS:", HAS_POLARS)
print("=" * 100)

# ------------------------------------------------------------
# 1. HTTP session with retry
# ------------------------------------------------------------

session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "FREIGHT-MNet academic research downloader "
            "(BTS Border Crossing Entry Data)"
        )
    }
)

retry = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=1.5,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
session.mount("http://", adapter)
session.mount("https://", adapter)

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def log_event(record: dict) -> None:
    record = dict(record)
    record["timestamp"] = pd.Timestamp.now().isoformat()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_size_mb(path: Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / 1024**2, 4)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def request_json(url: str, params: dict | None = None, timeout: int = 180):
    r = session.get(url, params=params, timeout=timeout)
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
    return r.json()


def build_urls(domain: str) -> dict:
    return {
        "domain": domain,
        "landing": f"{domain}/d/{DATASET_ID}",
        "metadata": f"{domain}/api/views/{DATASET_ID}.json",
        "columns": f"{domain}/api/views/{DATASET_ID}/columns.json",
        "full_csv": f"{domain}/api/views/{DATASET_ID}/rows.csv?accessType=DOWNLOAD",
        "resource_csv": f"{domain}/resource/{DATASET_ID}.csv",
        "resource_json": f"{domain}/resource/{DATASET_ID}.json",
        "count": f"{domain}/resource/{DATASET_ID}.json?$select=count(*)",
    }


def choose_working_domain() -> dict:
    last_error = None

    for domain in SOCRATA_DOMAINS:
        urls = build_urls(domain)
        try:
            meta = request_json(urls["metadata"], timeout=120)
            if isinstance(meta, dict) and meta.get("id") == DATASET_ID:
                print("[OK] Working Socrata domain:", domain)
                return urls
        except Exception as e:
            last_error = repr(e)
            print("[WARN] domain failed:", domain, last_error)

    raise RuntimeError(f"No working Socrata domain found for {DATASET_ID}. Last error: {last_error}")


def looks_like_valid_csv(path: Path, min_rows: int = 1) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        df = pd.read_csv(path, nrows=min(5, max(min_rows, 1)))
        return df.shape[1] > 0
    except Exception:
        return False


def looks_like_valid_parquet(path: Path) -> bool:
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    try:
        if HAS_POLARS:
            _ = pl.scan_parquet(str(path)).select(pl.len()).collect().item()
        else:
            _ = pd.read_parquet(path)
        return True
    except Exception:
        return False


def download_streaming_csv(url: str, out_path: Path, label: str, overwrite: bool = False) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and out_path.exists() and not overwrite and looks_like_valid_csv(out_path):
        print(f"[skip existing valid] {label}: {out_path.name} ({file_size_mb(out_path)} MB)")
        log_event({
            "status": "exists_valid",
            "url": url,
            "path": str(out_path),
            "size_mb": file_size_mb(out_path),
            "label": label,
        })
        return out_path

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    if part_path.exists():
        part_path.unlink()

    print("\n" + "-" * 100)
    print("[download]", label)
    print("URL:", url)
    print("OUT:", out_path)
    print("-" * 100)

    with session.get(url, stream=True, timeout=300) as r:
        if r.status_code >= 400:
            raise RuntimeError(f"HTTP {r.status_code} while downloading {url}\n{r.text[:500]}")

        total = int(r.headers.get("content-length", 0))
        with part_path.open("wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=label[:70],
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    part_path.replace(out_path)

    if not looks_like_valid_csv(out_path):
        raise RuntimeError(f"Downloaded file did not pass CSV validation: {out_path}")

    log_event({
        "status": "downloaded",
        "url": url,
        "path": str(out_path),
        "size_mb": file_size_mb(out_path),
        "label": label,
    })

    return out_path


def get_socrata_count(count_url: str) -> int:
    data = request_json(count_url, timeout=180)
    if isinstance(data, list) and len(data) > 0:
        first = data[0]
        if "count" in first:
            return int(first["count"])
        return int(list(first.values())[0])
    raise RuntimeError(f"Could not parse row count from response: {data}")


def save_metadata_and_columns(urls: dict) -> tuple[pd.DataFrame, dict]:
    metadata = request_json(urls["metadata"], timeout=120)
    columns = request_json(urls["columns"], timeout=120)

    METADATA_JSON_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    COLUMNS_JSON_PATH.write_text(json.dumps(columns, indent=2, ensure_ascii=False), encoding="utf-8")

    col_rows = []
    for c in columns:
        col_rows.append({
            "name": c.get("name"),
            "fieldName": c.get("fieldName"),
            "dataTypeName": c.get("dataTypeName"),
            "description": c.get("description"),
            "position": c.get("position"),
        })

    columns_df = pd.DataFrame(col_rows)
    columns_df.to_csv(COLUMNS_CSV_PATH, index=False, encoding="utf-8-sig")

    source_info = {
        "dataset_id": DATASET_ID,
        "domain": urls["domain"],
        "landing": urls["landing"],
        "metadata_url": urls["metadata"],
        "columns_url": urls["columns"],
        "full_csv_url": urls["full_csv"],
        "resource_csv_url": urls["resource_csv"],
        "metadata_name": metadata.get("name"),
        "metadata_description": metadata.get("description"),
        "metadata_rowsUpdatedAt": metadata.get("rowsUpdatedAt"),
        "metadata_viewLastModified": metadata.get("viewLastModified"),
        "metadata_totalTimesRated": metadata.get("totalTimesRated"),
    }
    SOURCE_INFO_PATH.write_text(json.dumps(source_info, indent=2, ensure_ascii=False), encoding="utf-8")

    print("[metadata saved]", METADATA_JSON_PATH)
    print("[columns saved]", COLUMNS_CSV_PATH)

    return columns_df, metadata


def infer_order_by_columns(columns_df: pd.DataFrame) -> str | None:
    """
    Build a stable-ish Socrata order-by clause from available field names.
    We do not assume the exact schema; we detect common field names.
    """
    field_names = set(columns_df["fieldName"].dropna().astype(str).tolist())

    preferred = [
        "date",
        "port_code",
        "port_name",
        "border",
        "state",
        "measure",
        "value",
    ]

    available = [c for c in preferred if c in field_names]

    if available:
        return ", ".join(available)

    # Fallback: use first few columns.
    fallback = list(field_names)[:3]
    if fallback:
        return ", ".join(fallback)

    return None


def query_chunk_csv(resource_csv_url: str, out_path: Path, limit: int, offset: int, order_by: str | None) -> dict:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and looks_like_valid_csv(out_path):
        n_rows = len(pd.read_csv(out_path))
        return {
            "status": "exists_valid",
            "offset": offset,
            "limit": limit,
            "path": str(out_path),
            "n_rows": n_rows,
            "size_mb": file_size_mb(out_path),
        }

    params = {
        "$limit": limit,
        "$offset": offset,
    }

    if order_by:
        params["$order"] = order_by

    print(f"[chunk] offset={offset:,}, limit={limit:,}")

    r = session.get(resource_csv_url, params=params, timeout=300)

    if r.status_code >= 400:
        # If order-by caused an error, retry without order.
        if order_by:
            print(f"[warn] chunk query failed with order_by={order_by!r}; retrying without order.")
            params.pop("$order", None)
            r = session.get(resource_csv_url, params=params, timeout=300)

    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    part_path = out_path.with_suffix(out_path.suffix + ".part")
    part_path.write_bytes(r.content)
    part_path.replace(out_path)

    if not looks_like_valid_csv(out_path):
        raise RuntimeError(f"Chunk does not look like valid CSV: {out_path}")

    n_rows = len(pd.read_csv(out_path))

    return {
        "status": "downloaded",
        "offset": offset,
        "limit": limit,
        "path": str(out_path),
        "n_rows": n_rows,
        "size_mb": file_size_mb(out_path),
    }


def csv_to_parquet(csv_path: Path, parquet_path: Path) -> dict:
    parquet_path = Path(parquet_path)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    if RESUME and looks_like_valid_parquet(parquet_path):
        if HAS_POLARS:
            n_rows = pl.scan_parquet(str(parquet_path)).select(pl.len()).collect().item()
        else:
            n_rows = len(pd.read_parquet(parquet_path))

        return {
            "parquet_status": "exists_valid",
            "parquet_path": str(parquet_path),
            "parquet_size_mb": file_size_mb(parquet_path),
            "parquet_rows": n_rows,
        }

    if HAS_POLARS:
        df = pl.read_csv(str(csv_path), infer_schema_length=10000, ignore_errors=True)
        n_rows = df.height
        tmp = parquet_path.with_suffix(".parquet.part")
        df.write_parquet(str(tmp), compression="zstd")
        tmp.replace(parquet_path)
        del df
    else:
        df = pd.read_csv(csv_path, low_memory=False)
        n_rows = len(df)
        tmp = parquet_path.with_suffix(".parquet.part")
        df.to_parquet(tmp, index=False)
        tmp.replace(parquet_path)
        del df

    return {
        "parquet_status": "converted",
        "parquet_path": str(parquet_path),
        "parquet_size_mb": file_size_mb(parquet_path),
        "parquet_rows": n_rows,
    }


def combine_chunks_to_dataframe(chunk_paths: list[Path]) -> pd.DataFrame:
    dfs = []
    for p in tqdm(chunk_paths, desc="Reading chunks"):
        dfs.append(pd.read_csv(p, low_memory=False))
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def normalize_column_lookup(df: pd.DataFrame) -> dict:
    return {c.lower().strip(): c for c in df.columns}


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lookup = normalize_column_lookup(df)
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]

    # Fuzzy fallback.
    norm = {re.sub(r"[^a-z0-9]+", "", k): v for k, v in lookup.items()}
    for c in candidates:
        key = re.sub(r"[^a-z0-9]+", "", c.lower())
        if key in norm:
            return norm[key]

    return None


def build_processed_outputs(full_df: pd.DataFrame) -> dict:
    """
    Creates:
      - measure inventory
      - 2018-2024 freight-only subset
      - monthly gateway pivot
    """
    result = {}

    if full_df.empty:
        return {"processed_status": "failed_empty_dataframe"}

    date_col = find_col(full_df, ["date", "Date"])
    measure_col = find_col(full_df, ["measure", "Measure"])
    value_col = find_col(full_df, ["value", "Value"])
    port_col = find_col(full_df, ["port_name", "Port Name", "Port"])
    state_col = find_col(full_df, ["state", "State"])
    border_col = find_col(full_df, ["border", "Border"])
    port_code_col = find_col(full_df, ["port_code", "Port Code"])

    required_cols = {
        "date": date_col,
        "measure": measure_col,
        "value": value_col,
    }

    missing_required = [k for k, v in required_cols.items() if v is None]
    if missing_required:
        return {
            "processed_status": "failed_missing_required_columns",
            "missing_required": missing_required,
            "available_columns": list(full_df.columns),
        }

    df = full_df.copy()

    df["_date"] = pd.to_datetime(df[date_col], errors="coerce")
    df["_year"] = df["_date"].dt.year
    df["_month"] = df["_date"].dt.month
    df["_value_numeric"] = pd.to_numeric(df[value_col], errors="coerce")

    # Measure inventory.
    measure_inventory = (
        df.groupby(measure_col, dropna=False)
        .agg(
            n_rows=(measure_col, "size"),
            min_year=("_year", "min"),
            max_year=("_year", "max"),
            total_value=("_value_numeric", "sum"),
        )
        .reset_index()
        .sort_values("n_rows", ascending=False)
    )
    measure_inventory.to_csv(MEASURE_INVENTORY_PATH, index=False, encoding="utf-8-sig")

    # Freight-relevant measures:
    # Keep trucks, truck containers, trains, rail containers.
    # Exclude passengers / pedestrians / personal vehicles / buses where possible.
    measure_text = df[measure_col].astype(str).str.lower()

    freight_mask = (
        measure_text.str.contains("truck", regex=False)
        | measure_text.str.contains("train", regex=False)
        | measure_text.str.contains("rail", regex=False)
    )

    exclude_mask = (
        measure_text.str.contains("passenger", regex=False)
        | measure_text.str.contains("pedestrian", regex=False)
        | measure_text.str.contains("bus", regex=False)
        | measure_text.str.contains("personal", regex=False)
        | measure_text.str.contains("privately owned", regex=False)
        | measure_text.str.contains("pov", regex=False)
    )

    core = df[
        df["_year"].isin(CORE_YEARS)
        & freight_mask
        & (~exclude_mask)
    ].copy()

    # Keep important columns plus any available location columns.
    keep_cols = []
    for c in [
        date_col,
        port_col,
        port_code_col,
        state_col,
        border_col,
        measure_col,
        value_col,
        "_date",
        "_year",
        "_month",
        "_value_numeric",
    ]:
        if c is not None and c not in keep_cols:
            keep_cols.append(c)

    # Add lat/lon if present.
    for c in ["latitude", "Latitude", "lat", "longitude", "Longitude", "lon", "Point", "point"]:
        fc = find_col(df, [c])
        if fc is not None and fc not in keep_cols:
            keep_cols.append(fc)

    core = core[keep_cols].copy()

    # Save core freight subset.
    core.to_csv(CORE_FREIGHT_CSV_PATH, index=False, encoding="utf-8-sig")
    core.to_parquet(CORE_FREIGHT_PARQUET_PATH, index=False)

    result["measure_inventory_path"] = str(MEASURE_INVENTORY_PATH)
    result["core_freight_csv_path"] = str(CORE_FREIGHT_CSV_PATH)
    result["core_freight_parquet_path"] = str(CORE_FREIGHT_PARQUET_PATH)
    result["core_freight_rows"] = len(core)

    # Create port-month-measure pivot if location columns exist.
    pivot_index = []
    for c in [date_col, port_col, port_code_col, state_col, border_col, "_year", "_month"]:
        if c is not None and c in core.columns and c not in pivot_index:
            pivot_index.append(c)

    if measure_col in core.columns and "_value_numeric" in core.columns and pivot_index:
        pivot = (
            core.pivot_table(
                index=pivot_index,
                columns=measure_col,
                values="_value_numeric",
                aggfunc="sum",
                fill_value=0,
            )
            .reset_index()
        )

        # Flatten columns.
        pivot.columns = [str(c).strip().replace(" ", "_").lower() for c in pivot.columns]
        pivot.to_parquet(MONTHLY_GATEWAY_PIVOT_PATH, index=False)

        result["monthly_gateway_pivot_path"] = str(MONTHLY_GATEWAY_PIVOT_PATH)
        result["monthly_gateway_pivot_rows"] = len(pivot)

    result["processed_status"] = "ok"
    return result


# ------------------------------------------------------------
# 3. Choose domain, save metadata, get count
# ------------------------------------------------------------

urls = choose_working_domain()
columns_df, metadata = save_metadata_and_columns(urls)

row_count = get_socrata_count(urls["count"])
print("Socrata row count:", row_count)

order_by = infer_order_by_columns(columns_df)
print("Order by clause:", order_by)

source_info = json.loads(SOURCE_INFO_PATH.read_text(encoding="utf-8"))
source_info.update({
    "row_count_from_socrata_api": row_count,
    "order_by_clause_used_for_chunks": order_by,
    "core_years_for_processed_subset": CORE_YEARS,
})
SOURCE_INFO_PATH.write_text(json.dumps(source_info, indent=2, ensure_ascii=False), encoding="utf-8")

# ------------------------------------------------------------
# 4. Download full raw CSV snapshot
# ------------------------------------------------------------

manifest_rows = []

if RESUME and MANIFEST_PATH.exists():
    try:
        old_manifest = pd.read_csv(MANIFEST_PATH)
        manifest_rows = old_manifest.to_dict("records")
        print(f"Loaded previous manifest with {len(manifest_rows)} rows.")
    except Exception:
        manifest_rows = []

if DOWNLOAD_FULL_SNAPSHOT:
    rec = {
        "record_type": "full_snapshot",
        "dataset_id": DATASET_ID,
        "url": urls["full_csv"],
        "out_path": str(FULL_CSV_PATH),
    }

    try:
        p = download_streaming_csv(urls["full_csv"], FULL_CSV_PATH, label="BTS Border Crossing full CSV snapshot")
        rec["status"] = "downloaded_or_exists"
        rec["size_mb"] = file_size_mb(p)
        if COMPUTE_SHA256:
            rec["sha256"] = sha256_file(p)

        # Save preview.
        preview = pd.read_csv(p, nrows=50, low_memory=False)
        preview.to_csv(PREVIEW_PATH, index=False, encoding="utf-8-sig")
        rec["preview_path"] = str(PREVIEW_PATH)
        rec["preview_columns"] = "|".join(preview.columns)

        # Convert full snapshot to Parquet.
        if CONVERT_TO_PARQUET:
            pq_info = csv_to_parquet(p, FULL_PARQUET_PATH)
            rec.update(pq_info)

    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = repr(e)
        rec["traceback"] = traceback.format_exc()[:4000]
        print("[FAILED] full snapshot:", e)

    manifest_rows.append(rec)
    pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5. Download resumable API chunks
# ------------------------------------------------------------

chunk_records = []

if DOWNLOAD_CHUNKS:
    offsets = list(range(0, row_count, CHUNK_SIZE))

    for offset in tqdm(offsets, desc="Downloading Socrata chunks"):
        chunk_idx = offset // CHUNK_SIZE
        chunk_csv_path = DIRS["chunks_csv"] / f"border_crossing_{DATASET_ID}_chunk_{chunk_idx:05d}_offset_{offset}.csv"
        chunk_parquet_path = DIRS["chunks_parquet"] / f"border_crossing_{DATASET_ID}_chunk_{chunk_idx:05d}_offset_{offset}.parquet"

        rec = {
            "record_type": "chunk",
            "dataset_id": DATASET_ID,
            "chunk_idx": chunk_idx,
            "offset": offset,
            "limit": CHUNK_SIZE,
            "csv_path": str(chunk_csv_path),
            "parquet_path": str(chunk_parquet_path),
        }

        try:
            chunk_info = query_chunk_csv(
                resource_csv_url=urls["resource_csv"],
                out_path=chunk_csv_path,
                limit=CHUNK_SIZE,
                offset=offset,
                order_by=order_by,
            )
            rec.update(chunk_info)

            if CONVERT_TO_PARQUET:
                pq_info = csv_to_parquet(chunk_csv_path, chunk_parquet_path)
                rec.update(pq_info)

        except Exception as e:
            rec["status"] = "failed"
            rec["error"] = repr(e)
            rec["traceback"] = traceback.format_exc()[:3000]
            print(f"[FAILED] chunk offset={offset}: {e}")

        chunk_records.append(rec)

        # Save progress after every chunk.
        all_rows = manifest_rows + chunk_records
        pd.DataFrame(all_rows).to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

        time.sleep(0.15)

# ------------------------------------------------------------
# 6. Build processed outputs
# ------------------------------------------------------------

processed_rec = {
    "record_type": "processed",
    "dataset_id": DATASET_ID,
}

try:
    # Prefer full snapshot if it exists. Otherwise combine chunks.
    if FULL_CSV_PATH.exists() and looks_like_valid_csv(FULL_CSV_PATH):
        full_df = pd.read_csv(FULL_CSV_PATH, low_memory=False)
        processed_rec["source_used_for_processing"] = str(FULL_CSV_PATH)
    else:
        chunk_paths = sorted(DIRS["chunks_csv"].glob(f"border_crossing_{DATASET_ID}_chunk_*.csv"))
        full_df = combine_chunks_to_dataframe(chunk_paths)
        processed_rec["source_used_for_processing"] = "chunks_csv"

    processed_rec["full_rows_loaded_for_processing"] = len(full_df)
    processed_rec["full_columns"] = "|".join(full_df.columns)

    if SAVE_CORE_2018_2024_FREIGHT_SUBSET:
        processed_info = build_processed_outputs(full_df)
        processed_rec.update(processed_info)

except Exception as e:
    processed_rec["status"] = "failed"
    processed_rec["error"] = repr(e)
    processed_rec["traceback"] = traceback.format_exc()[:4000]
    print("[FAILED] processed outputs:", e)

manifest_rows = manifest_rows + chunk_records + [processed_rec]

manifest = pd.DataFrame(manifest_rows)

# Deduplicate reruns.
dedup_cols = [c for c in ["record_type", "dataset_id", "url", "out_path", "chunk_idx", "csv_path"] if c in manifest.columns]
if dedup_cols:
    manifest = manifest.drop_duplicates(subset=dedup_cols, keep="last").reset_index(drop=True)

manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("BTS BORDER CROSSING ENTRY DATA DOWNLOAD COMPLETE")
print("=" * 100)
print("BORDER_ROOT:", BORDER_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Log:", LOG_PATH)
print("Metadata:", METADATA_JSON_PATH)
print("Columns:", COLUMNS_CSV_PATH)
print("Full CSV:", FULL_CSV_PATH)
print("Full Parquet:", FULL_PARQUET_PATH)
print("Core freight subset Parquet:", CORE_FREIGHT_PARQUET_PATH)
print("=" * 100)

display_cols = [
    "record_type",
    "status",
    "size_mb",
    "parquet_status",
    "parquet_rows",
    "offset",
    "n_rows",
    "out_path",
    "csv_path",
    "parquet_path",
]
display_cols = [c for c in display_cols if c in manifest.columns]
display(manifest[display_cols].tail(30))

print("\nSource info:")
print(json.dumps(source_info, indent=2, ensure_ascii=False))

if MEASURE_INVENTORY_PATH.exists():
    print("\nMeasure inventory:")
    measure_inventory = pd.read_csv(MEASURE_INVENTORY_PATH)
    display(measure_inventory.head(30))

if CORE_FREIGHT_PARQUET_PATH.exists():
    print("\nCore 2018–2024 freight subset preview:")
    core_preview = pd.read_parquet(CORE_FREIGHT_PARQUET_PATH)
    display(core_preview.head(20))
    print("Rows:", len(core_preview))
    print("Columns:", list(core_preview.columns))

BTS Border Crossing Entry Data downloader
DATA_ROOT: E:\NetworkOptimization\pythonProject1\Data
BORDER_ROOT: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing
DATASET_ID: keg4-3bc2
CORE_YEARS: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
HAS_POLARS: False
[OK] Working Socrata domain: https://data.bts.gov
[metadata saved] E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\_metadata\border_crossing_entry_data_keg4-3bc2_metadata.json
[columns saved] E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\_metadata\border_crossing_entry_data_keg4-3bc2_columns.csv
Socrata row count: 273891
Order by clause: date, port_code, port_name, border, state, measure, value

----------------------------------------------------------------------------------------------------
[download] BTS Border Crossing full CSV snapshot
URL: https://data.bts.gov/api/views/keg4-3bc2/rows.csv?accessType=DOWNLOAD
OUT: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\raw_full_snap

BTS Border Crossing full CSV snapshot: 0.00B [00:00, ?B/s]

[chunk] offset=0, limit=50,000
[chunk] offset=50,000, limit=50,000
[chunk] offset=100,000, limit=50,000
[chunk] offset=150,000, limit=50,000
[chunk] offset=200,000, limit=50,000
[chunk] offset=250,000, limit=50,000


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_32072\1272047040.py:545: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["_date"] = pd.to_datetime(df[date_col], errors="coerce")



BTS BORDER CROSSING ENTRY DATA DOWNLOAD COMPLETE
BORDER_ROOT: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing
Manifest: E:\NetworkOptimization\pythonProject1\Data\00_manifest\bts_border_crossing_download_manifest.csv
Log: E:\NetworkOptimization\pythonProject1\Data\00_manifest\bts_border_crossing_download_log.jsonl
Metadata: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\_metadata\border_crossing_entry_data_keg4-3bc2_metadata.json
Columns: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\_metadata\border_crossing_entry_data_keg4-3bc2_columns.csv
Full CSV: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\raw_full_snapshot\border_crossing_entry_data_keg4-3bc2_full.csv
Full Parquet: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\parquet\border_crossing_entry_data_keg4-3bc2_full.parquet
Core freight subset Parquet: E:\NetworkOptimization\pythonProject1\Data\06_border_crossing\processed\border_crossing_freight_measures_20

...


Source info:
{
  "dataset_id": "keg4-3bc2",
  "domain": "https://data.bts.gov",
  "landing": "https://data.bts.gov/d/keg4-3bc2",
  "metadata_url": "https://data.bts.gov/api/views/keg4-3bc2.json",
  "columns_url": "https://data.bts.gov/api/views/keg4-3bc2/columns.json",
  "full_csv_url": "https://data.bts.gov/api/views/keg4-3bc2/rows.csv?accessType=DOWNLOAD",
  "resource_csv_url": "https://data.bts.gov/resource/keg4-3bc2.csv",
  "metadata_name": "Border Crossing Entry Data",
  "metadata_description": "The Bureau of Transportation Statistics (BTS) Border Crossing Data provide summary statistics for inbound crossings at the U.S.-Canada and the U.S.-Mexico border at the port level.  Data are available for trucks, trains, buses, personal vehicles, passengers, and pedestrians.  Border crossing data are collected at ports of entry by U.S. Customs and Border Protection (CBP).  The data reflect the number of vehicles, containers, passengers or pedestrians entering the United States.  CBP does 

,Measure,n_rows,min_year,max_year,total_value
0,Personal Vehicles,39190,1996,2026,3155947792
1,Personal Vehicle Passengers,39165,1996,2026,6520236589
2,Trucks,37964,1996,2026,341681707
3,Pedestrians,33306,1996,2026,1307746405
4,Buses,32231,1996,2026,9553775
5,Bus Passengers,32212,1996,2026,160206184
6,Trains,30227,1996,2026,1123143
7,Train Passengers,29596,1996,2026,7535981



Core 2018–2024 freight subset preview:


,Date,Port Name,Port Code,State,Border,Measure,Value,_date,_year,_month,_value_numeric,Latitude,Longitude,Point
0,Apr 2021,Otay Mesa,2506,California,US-Mexico Border,Trains,16,2021-04-01,2021,4,16,32.550,-116.939,POINT (-116.938538 32.55033)
1,Oct 2023,Otay Mesa,2506,California,US-Mexico Border,Trucks,92196,2023-10-01,2023,10,92196,32.550,-116.939,POINT (-116.938538 32.55033)
2,Dec 2021,Alexandria Bay,708,New York,US-Canada Border,Trucks,14090,2021-12-01,2021,12,14090,44.347,-75.984,POINT (-75.983592 44.347229)
3,Nov 2023,Wildhorse,3323,Montana,US-Canada Border,Trucks,9,2023-11-01,2023,11,9,48.999,-110.215,POINT (-110.215083 48.999361)
4,May 2024,International Falls,3604,Minnesota,US-Canada Border,Trucks,1414,2024-05-01,2024,5,1414,48.608,-93.401,POINT (-93.401355 48.6078)
5,Apr 2022,Roosville,3318,Montana,US-Canada Border,Trucks,707,2022-04-01,2022,4,707,49.000,-115.056,POINT (-115.056027 48.999638)
6,Nov 2021,Calais,115,Maine,US-Canada Border,Trucks,5753,2021-11-01,2021,11,5753,45.189,-67.275,POINT (-67.275381 45.188548)
7,Jul 2023,Otay Mesa,2506,California,US-Mexico Border,Trucks,89332,2023-07-01,2023,7,89332,32.550,-116.939,POINT (-116.938538 32.55033)
8,Dec 2024,Portal,3403,North Dakota,US-Canada Border,Trains,119,2024-12-01,2024,12,119,48.999,-102.552,POINT (-102.551944 48.998944)
9,Nov 2022,Skagway,3103,Alaska,US-Canada Border,Trucks,192,2022-11-01,2022,11,192,59.630,-135.164,POINT (-135.164444 59.629722)


Rows: 10605
Columns: ['Date', 'Port Name', 'Port Code', 'State', 'Border', 'Measure', 'Value', '_date', '_year', '_month', '_value_numeric', 'Latitude', 'Longitude', 'Point']


In [4]:
# ============================================================
# Repair pyarrow.parquet / pandas residue for FREIGHT-MNet env
# Run this once, then RESTART kernel.
# ============================================================

import sys
import site
import shutil
import subprocess
from pathlib import Path

print("=" * 100)
print("Current Python executable:")
print(sys.executable)
print("=" * 100)

site_packages = Path(site.getsitepackages()[0])
print("site-packages:", site_packages)

# 1. Clean broken leftover folders, especially ~andas.
for pattern in ["~andas*", "~umpy*", "~andas-*.dist-info", "~umpy-*.dist-info"]:
    for p in site_packages.glob(pattern):
        print("Removing broken leftover:", p)
        try:
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
        except Exception as e:
            print("  failed:", e)

# 2. Reinstall pandas and pyarrow cleanly.
# pandas==2.3.3 is safer because copulas requires pandas<3.
cmds = [
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "--prefer-binary",
        "pandas==2.3.3",
        "pyarrow",
    ],
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade",
        "--no-cache-dir",
        "--prefer-binary",
        "fastparquet",
    ],
]

for cmd in cmds:
    print("\n" + "-" * 100)
    print("$", " ".join(cmd))
    print("-" * 100)
    subprocess.run(cmd, check=True)

print("\n" + "=" * 100)
print("Repair install finished.")
print("IMPORTANT: Restart the Jupyter kernel now, then run the verification cell below.")
print("=" * 100)

Current Python executable:
D:\apphome\anaconda3\python.exe
site-packages: D:\apphome\anaconda3

----------------------------------------------------------------------------------------------------
$ D:\apphome\anaconda3\python.exe -m pip install --upgrade --force-reinstall --no-cache-dir --prefer-binary pandas==2.3.3 pyarrow
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
$ D:\apphome\anaconda3\python.exe -m pip install --upgrade --no-cache-dir --prefer-binary fastparquet
----------------------------------------------------------------------------------------------------

Repair install finished.
IMPORTANT: Restart the Jupyter kernel now, then run the verification cell below.


In [1]:
# ============================================================
# FREIGHT-MNet: Full Data Inventory Console Printer
#
# Goal:
#   Traverse every subfolder and dataset under:
#       E:\NetworkOptimization\pythonProject1\Data
#
# Prints to console:
#   1. Root summary
#   2. Top-level folder summary
#   3. Expected dataset checklist
#   4. Full directory tree
#   5. Full file inventory with absolute paths
#
# Also saves:
#   E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_console_output.txt
#   E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_file_inventory.csv
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import os
import sys
import csv
import fnmatch
import platform

# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
MANIFEST_DIR = DATA_ROOT / "00_manifest"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TXT = MANIFEST_DIR / "data_inventory_console_output.txt"
OUTPUT_CSV = MANIFEST_DIR / "data_inventory_file_inventory.csv"

# Print all directories and all files.
PRINT_ALL_DIRECTORIES = True
PRINT_ALL_FILES = True

# Set to None to print all files.
# If output is too long, you can set e.g. MAX_FULL_FILE_LINES = 5000.
MAX_FULL_FILE_LINES = None

# Directory tree depth:
# None means print all directory levels.
TREE_MAX_DEPTH = None

# Extensions we consider important data artifacts.
IMPORTANT_EXTENSIONS = {
    ".csv", ".csv.gz", ".parquet", ".zip", ".gz",
    ".xlsx", ".xls", ".pdf",
    ".json", ".jsonl", ".geojson", ".gpkg",
    ".shp", ".shx", ".dbf", ".prj", ".cpg",
    ".txt", ".dat", ".mdb", ".accdb",
}

TEMP_OR_INCOMPLETE_SUFFIXES = {
    ".part", ".tmp", ".temp", ".crdownload"
}

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def size_human(num_bytes: int) -> str:
    if num_bytes is None:
        return "NA"
    x = float(num_bytes)
    units = ["B", "KB", "MB", "GB", "TB"]
    for u in units:
        if x < 1024 or u == units[-1]:
            if u == "B":
                return f"{int(x)} {u}"
            return f"{x:.3f} {u}"
        x /= 1024

def size_mb(num_bytes: int) -> float:
    return round((num_bytes or 0) / 1024**2, 6)

def rel_path(p: Path) -> str:
    try:
        return str(p.relative_to(DATA_ROOT))
    except Exception:
        return str(p)

def abs_path(p: Path) -> str:
    return str(p.resolve())

def file_type(path: Path) -> str:
    name = path.name.lower()

    if name.endswith(".csv.gz"):
        return ".csv.gz"
    if name.endswith(".tar.gz"):
        return ".tar.gz"

    return path.suffix.lower() if path.suffix else "<no_ext>"

def is_temp_or_incomplete(path: Path) -> bool:
    name = path.name.lower()
    return any(name.endswith(s) for s in TEMP_OR_INCOMPLETE_SUFFIXES)

def get_mtime(path: Path) -> str:
    try:
        return datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return "NA"

def get_size(path: Path) -> int:
    try:
        return path.stat().st_size
    except Exception:
        return 0

def is_under(path: Path, parent: Path) -> bool:
    try:
        path.relative_to(parent)
        return True
    except Exception:
        return False

def matches_any_pattern(path: Path, patterns):
    if patterns is None:
        return True

    name = path.name
    rel = rel_path(path).replace("\\", "/")

    for pat in patterns:
        if fnmatch.fnmatch(name, pat) or fnmatch.fnmatch(rel, pat):
            return True

    return False

def path_depth(path: Path) -> int:
    try:
        return len(path.relative_to(DATA_ROOT).parts)
    except Exception:
        return 0

# ------------------------------------------------------------
# 2. Scan Data root
# ------------------------------------------------------------

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

all_dirs = []
all_files = []

for root, dirs, files in os.walk(DATA_ROOT):
    root_path = Path(root)

    # Stable order.
    dirs.sort()
    files.sort()

    all_dirs.append(root_path)

    for fname in files:
        p = root_path / fname
        if p.is_file():
            all_files.append(p)

all_dirs = sorted(set(all_dirs), key=lambda p: str(p).lower())
all_files = sorted(set(all_files), key=lambda p: str(p).lower())

# File metadata.
file_rows = []

for p in all_files:
    try:
        st = p.stat()
        size_bytes = st.st_size
        mtime = datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        size_bytes = 0
        mtime = "NA"

    rel = rel_path(p)
    parts = Path(rel).parts
    top_level = parts[0] if len(parts) > 0 else "<root>"

    file_rows.append({
        "absolute_path": abs_path(p),
        "relative_path": rel,
        "parent_dir": abs_path(p.parent),
        "top_level": top_level,
        "file_name": p.name,
        "extension": file_type(p),
        "size_bytes": size_bytes,
        "size_mb": size_mb(size_bytes),
        "modified_time": mtime,
        "is_temp_or_incomplete": is_temp_or_incomplete(p),
    })

total_bytes = sum(r["size_bytes"] for r in file_rows)

# Directory immediate file counts.
files_by_parent = defaultdict(list)
for p in all_files:
    files_by_parent[p.parent].append(p)

# Top-level stats.
top_stats = defaultdict(lambda: {
    "n_dirs": 0,
    "n_files": 0,
    "size_bytes": 0,
    "extensions": Counter(),
    "n_temp_or_incomplete": 0,
})

for d in all_dirs:
    rel = rel_path(d)
    parts = Path(rel).parts
    top = parts[0] if len(parts) > 0 and rel != "." else "<root>"
    top_stats[top]["n_dirs"] += 1

for r in file_rows:
    top = r["top_level"]
    top_stats[top]["n_files"] += 1
    top_stats[top]["size_bytes"] += r["size_bytes"]
    top_stats[top]["extensions"][r["extension"]] += 1
    if r["is_temp_or_incomplete"]:
        top_stats[top]["n_temp_or_incomplete"] += 1

# ------------------------------------------------------------
# 3. Prepare output writer
# ------------------------------------------------------------

out_f = OUTPUT_TXT.open("w", encoding="utf-8")

def emit(line=""):
    print(line)
    out_f.write(str(line) + "\n")

try:
    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    emit("=" * 120)
    emit("FREIGHT-MNet DATA INVENTORY")
    emit("=" * 120)
    emit(f"Run time:                  {now_str()}")
    emit(f"Python executable:         {sys.executable}")
    emit(f"Python version:            {sys.version.replace(chr(10), ' ')}")
    emit(f"Platform:                  {platform.platform()}")
    emit(f"DATA_ROOT:                 {abs_path(DATA_ROOT)}")
    emit(f"Output TXT:                {abs_path(OUTPUT_TXT)}")
    emit(f"Output CSV:                {abs_path(OUTPUT_CSV)}")
    emit("-" * 120)
    emit(f"Total directories:         {len(all_dirs):,}")
    emit(f"Total files:               {len(all_files):,}")
    emit(f"Total size:                {size_human(total_bytes)}")
    emit("=" * 120)

    # --------------------------------------------------------
    # Global extension summary
    # --------------------------------------------------------

    ext_counter = Counter(r["extension"] for r in file_rows)
    temp_files = [r for r in file_rows if r["is_temp_or_incomplete"]]

    emit("")
    emit("=" * 120)
    emit("GLOBAL FILE TYPE SUMMARY")
    emit("=" * 120)
    for ext, n in sorted(ext_counter.items(), key=lambda x: (-x[1], x[0])):
        ext_size = sum(r["size_bytes"] for r in file_rows if r["extension"] == ext)
        emit(f"{ext:12s} | files={n:8,} | size={size_human(ext_size):>15s}")

    emit("")
    emit(f"Temporary / incomplete files detected: {len(temp_files):,}")
    if temp_files:
        emit("TEMP / INCOMPLETE FILES:")
        for r in temp_files[:100]:
            emit(f"  {r['extension']:10s} | {size_human(r['size_bytes']):>12s} | {r['absolute_path']}")
        if len(temp_files) > 100:
            emit(f"  ... {len(temp_files) - 100:,} more temp/incomplete files not shown here.")

    # --------------------------------------------------------
    # Top-level folder summary
    # --------------------------------------------------------

    emit("")
    emit("=" * 120)
    emit("TOP-LEVEL FOLDER SUMMARY")
    emit("=" * 120)
    emit(f"{'TOP_FOLDER':35s} | {'DIRS':>8s} | {'FILES':>8s} | {'TEMP':>6s} | {'SIZE':>15s} | EXTENSIONS")
    emit("-" * 120)

    for top in sorted(top_stats.keys()):
        s = top_stats[top]
        ext_summary = ", ".join(
            f"{k}:{v}" for k, v in sorted(s["extensions"].items(), key=lambda x: (-x[1], x[0]))[:10]
        )
        emit(
            f"{top:35s} | "
            f"{s['n_dirs']:8,} | "
            f"{s['n_files']:8,} | "
            f"{s['n_temp_or_incomplete']:6,} | "
            f"{size_human(s['size_bytes']):>15s} | "
            f"{ext_summary}"
        )

    # --------------------------------------------------------
    # Expected dataset checklist
    # --------------------------------------------------------

    dataset_checks = []

    def add_check(label, path, patterns=None):
        dataset_checks.append({
            "label": label,
            "path": Path(path),
            "patterns": patterns,
        })

    # Core folders.
    add_check("00_manifest", DATA_ROOT / "00_manifest")
    add_check("01_faf", DATA_ROOT / "01_faf")
    add_check("02_fmi", DATA_ROOT / "02_fmi")
    add_check("03_ntad_geospatial", DATA_ROOT / "03_ntad_geospatial")
    add_check("04_stb", DATA_ROOT / "04_stb")
    add_check("05_noaa_storm_events", DATA_ROOT / "05_noaa_storm_events")
    add_check("06_border_crossing", DATA_ROOT / "06_border_crossing")
    add_check("07_census_boundaries", DATA_ROOT / "07_census_boundaries")
    add_check("08_processed", DATA_ROOT / "08_processed")
    add_check("09_crosswalks", DATA_ROOT / "09_crosswalks")

    # FAF.
    add_check("FAF raw_zip", DATA_ROOT / "01_faf" / "raw_zip", ["*.zip"])
    add_check("FAF flows_regional", DATA_ROOT / "01_faf" / "flows_regional")
    add_check("FAF flows_state", DATA_ROOT / "01_faf" / "flows_state")
    add_check("FAF flows_regional_full", DATA_ROOT / "01_faf" / "flows_regional_full")
    add_check("FAF access_database", DATA_ROOT / "01_faf" / "access_database")
    add_check("FAF county_experimental", DATA_ROOT / "01_faf" / "county_experimental")
    add_check("FAF lookup_tables", DATA_ROOT / "01_faf" / "lookup_tables")

    # FMI by year.
    for y in range(2018, 2025):
        add_check(f"FMI {y}", DATA_ROOT / "02_fmi" / str(y), ["*.csv", "*.parquet", "*.json"])
    add_check("FMI metadata", DATA_ROOT / "02_fmi" / "_metadata")

    # NTAD layers.
    for layer in [
        "faf_regions",
        "faf_network_links",
        "faf_network_nodes",
        "narn_lines",
        "narn_nodes",
        "intermodal_rail_tofc_cofc",
    ]:
        add_check(f"NTAD {layer}", DATA_ROOT / "03_ntad_geospatial" / layer)
        add_check(f"NTAD {layer} parquet_chunks_v2", DATA_ROOT / "03_ntad_geospatial" / layer / "parquet_chunks_v2", ["*.parquet"])
        add_check(f"NTAD {layer} metadata_v2", DATA_ROOT / "03_ntad_geospatial" / layer / "metadata_v2")

    # STB Waybill by year.
    for y in range(2018, 2025):
        add_check(f"STB Waybill {y}", DATA_ROOT / "04_stb" / "waybill_public_use" / str(y))
        add_check(f"STB Waybill {y} sample_zip", DATA_ROOT / "04_stb" / "waybill_public_use" / str(y) / "sample_zip", ["*.zip"])
        add_check(f"STB Waybill {y} reference_guide", DATA_ROOT / "04_stb" / "waybill_public_use" / str(y) / "reference_guide", ["*.pdf"])
        add_check(f"STB Waybill {y} extracted", DATA_ROOT / "04_stb" / "waybill_public_use" / str(y) / "extracted")
        add_check(f"STB Waybill {y} preview", DATA_ROOT / "04_stb" / "waybill_public_use" / str(y) / "preview")

    # STB Rail Service.
    add_check("STB Rail Service root", DATA_ROOT / "04_stb" / "rail_service_performance")
    add_check("STB Rail weekly_service_files", DATA_ROOT / "04_stb" / "rail_service_performance" / "weekly_service_files")
    add_check("STB Rail STB-1145 CSV", DATA_ROOT / "04_stb" / "rail_service_performance" / "stb_1145_csv")
    add_check("STB Rail methodology_docs", DATA_ROOT / "04_stb" / "rail_service_performance" / "methodology_docs")
    add_check("STB Rail metadata", DATA_ROOT / "04_stb" / "rail_service_performance" / "_metadata")

    # NOAA.
    for table in ["details", "locations", "fatalities"]:
        add_check(f"NOAA {table} raw", DATA_ROOT / "05_noaa_storm_events" / table, ["*.gz", "*.csv.gz"])
        add_check(f"NOAA {table} parquet", DATA_ROOT / "05_noaa_storm_events" / "parquet" / table, ["*.parquet"])
        add_check(f"NOAA {table} preview", DATA_ROOT / "05_noaa_storm_events" / "preview" / table)
    add_check("NOAA docs", DATA_ROOT / "05_noaa_storm_events" / "docs")
    add_check("NOAA metadata", DATA_ROOT / "05_noaa_storm_events" / "_metadata")

    # Border.
    add_check("Border root", DATA_ROOT / "06_border_crossing")
    add_check("Border raw full snapshot", DATA_ROOT / "06_border_crossing" / "raw_full_snapshot")
    add_check("Border chunks_csv", DATA_ROOT / "06_border_crossing" / "chunks_csv")
    add_check("Border chunks_parquet", DATA_ROOT / "06_border_crossing" / "chunks_parquet")
    add_check("Border parquet", DATA_ROOT / "06_border_crossing" / "parquet")
    add_check("Border processed", DATA_ROOT / "06_border_crossing" / "processed")
    add_check("Border metadata", DATA_ROOT / "06_border_crossing" / "_metadata")

    # Census.
    add_check("Census counties_2024 root", DATA_ROOT / "07_census_boundaries" / "counties_2024")
    add_check("Census raw_zip", DATA_ROOT / "07_census_boundaries" / "counties_2024" / "raw_zip")
    add_check("Census extracted", DATA_ROOT / "07_census_boundaries" / "counties_2024" / "extracted")
    add_check("Census geoparquet", DATA_ROOT / "07_census_boundaries" / "counties_2024" / "geoparquet")
    add_check("Census processed", DATA_ROOT / "07_census_boundaries" / "counties_2024" / "processed")
    add_check("Census metadata", DATA_ROOT / "07_census_boundaries" / "counties_2024" / "_metadata")

    emit("")
    emit("=" * 120)
    emit("EXPECTED DATASET CHECKLIST")
    emit("=" * 120)

    for item in dataset_checks:
        label = item["label"]
        p = item["path"]
        patterns = item["patterns"]

        exists = p.exists()

        if exists and p.is_dir():
            matched = [
                f for f in all_files
                if is_under(f, p) and matches_any_pattern(f, patterns)
            ]
            all_under = [
                f for f in all_files
                if is_under(f, p)
            ]
        elif exists and p.is_file():
            matched = [p] if matches_any_pattern(p, patterns) else []
            all_under = [p]
        else:
            matched = []
            all_under = []

        matched_size = sum(get_size(f) for f in matched)
        all_size = sum(get_size(f) for f in all_under)

        ext_counts = Counter(file_type(f) for f in matched)
        ext_summary = ", ".join(f"{k}:{v}" for k, v in sorted(ext_counts.items(), key=lambda x: (-x[1], x[0]))[:8])

        status = "OK" if exists and len(matched if patterns else all_under) > 0 else ("EMPTY" if exists else "MISSING")

        emit("-" * 120)
        emit(f"DATASET: {label}")
        emit(f"STATUS:  {status}")
        emit(f"PATH:    {abs_path(p)}")
        emit(f"FILES_MATCHED: {len(matched):,} | FILES_ALL_UNDER: {len(all_under):,}")
        emit(f"SIZE_MATCHED:  {size_human(matched_size)} | SIZE_ALL_UNDER: {size_human(all_size)}")
        emit(f"EXTENSIONS: {ext_summary if ext_summary else 'NA'}")

        # Representative files: first 3 and largest 3.
        if matched:
            emit("REPRESENTATIVE FILES:")
            for f in sorted(matched, key=lambda x: str(x).lower())[:3]:
                emit(f"  FIRST   | {size_human(get_size(f)):>12s} | {abs_path(f)}")

            largest = sorted(matched, key=lambda x: get_size(x), reverse=True)[:3]
            for f in largest:
                emit(f"  LARGEST | {size_human(get_size(f)):>12s} | {abs_path(f)}")

    # --------------------------------------------------------
    # Directory tree
    # --------------------------------------------------------

    if PRINT_ALL_DIRECTORIES:
        emit("")
        emit("=" * 120)
        emit("FULL DIRECTORY TREE")
        emit("=" * 120)

        for d in all_dirs:
            depth = path_depth(d)

            if TREE_MAX_DEPTH is not None and depth > TREE_MAX_DEPTH:
                continue

            rel = rel_path(d)
            indent = "  " * max(depth - 1, 0)
            immediate_files = files_by_parent.get(d, [])
            immediate_size = sum(get_size(f) for f in immediate_files)

            emit(
                f"{indent}[DIR] {rel} | "
                f"immediate_files={len(immediate_files):,} | "
                f"immediate_size={size_human(immediate_size)} | "
                f"absolute={abs_path(d)}"
            )

    # --------------------------------------------------------
    # Full file inventory
    # --------------------------------------------------------

    if PRINT_ALL_FILES:
        emit("")
        emit("=" * 120)
        emit("FULL FILE INVENTORY")
        emit("=" * 120)
        emit("FORMAT:")
        emit("INDEX | EXTENSION | SIZE | MODIFIED_TIME | TEMP_FLAG | ABSOLUTE_PATH")
        emit("-" * 120)

        rows_to_print = file_rows
        if MAX_FULL_FILE_LINES is not None:
            rows_to_print = file_rows[:MAX_FULL_FILE_LINES]

        for idx, r in enumerate(rows_to_print, start=1):
            emit(
                f"{idx:08d} | "
                f"{r['extension']:12s} | "
                f"{size_human(r['size_bytes']):>12s} | "
                f"{r['modified_time']} | "
                f"temp={str(r['is_temp_or_incomplete']):5s} | "
                f"{r['absolute_path']}"
            )

        if MAX_FULL_FILE_LINES is not None and len(file_rows) > MAX_FULL_FILE_LINES:
            emit(f"... output truncated by MAX_FULL_FILE_LINES={MAX_FULL_FILE_LINES}.")
            emit(f"... total files available: {len(file_rows):,}")

    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    with OUTPUT_CSV.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "absolute_path",
                "relative_path",
                "parent_dir",
                "top_level",
                "file_name",
                "extension",
                "size_bytes",
                "size_mb",
                "modified_time",
                "is_temp_or_incomplete",
            ]
        )
        writer.writeheader()
        writer.writerows(file_rows)

    # --------------------------------------------------------
    # Footer
    # --------------------------------------------------------

    emit("")
    emit("=" * 120)
    emit("INVENTORY COMPLETE")
    emit("=" * 120)
    emit(f"Total directories: {len(all_dirs):,}")
    emit(f"Total files:       {len(all_files):,}")
    emit(f"Total size:        {size_human(total_bytes)}")
    emit(f"Output TXT:        {abs_path(OUTPUT_TXT)}")
    emit(f"Output CSV:        {abs_path(OUTPUT_CSV)}")
    emit("=" * 120)

finally:
    out_f.close()

FREIGHT-MNet DATA INVENTORY
Run time:                  2026-05-16 16:32:12
Python executable:         E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version:            3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Platform:                  Windows-10-10.0.26200-SP0
DATA_ROOT:                 E:\NetworkOptimization\pythonProject1\Data
Output TXT:                E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_console_output.txt
Output CSV:                E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_file_inventory.csv
------------------------------------------------------------------------------------------------------------------------
Total directories:         277
Total files:               28,981
Total size:                15.074 GB

GLOBAL FILE TYPE SUMMARY
.json        | files=  22,234 | size=       4.346 GB
.geojson     | files=   3,836 | s

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)




INVENTORY COMPLETE
Total directories: 277
Total files:       28,981
Total size:        15.074 GB
Output TXT:        E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_console_output.txt
Output CSV:        E:\NetworkOptimization\pythonProject1\Data\00_manifest\data_inventory_file_inventory.csv
